In [0]:
%pip install openpyxl pymc arviz
%pip install pymc blackjax numpyro jax jaxlib

In [0]:
import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
import pytensor
import pytensor.tensor as pt
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_percentage_error, r2_score
import pymc as pm
import arviz as az
import pymc.sampling.jax as pmjax

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

print(f"PyMC  : {pm.__version__}")
print(f"ArviZ : {az.__version__}")

In [0]:
FILE_PATH = r"/Volumes/data_exploration_hub/marketing_hub/model_data/Final_Data/EXH_ModelData.xlsx"

df = pd.read_excel(FILE_PATH, sheet_name=0)
df = df.sort_values("week_date").reset_index(drop=True)

# Binary encoding
df["stat_holiday_in_week"] = (df["stat_holiday_in_week"] == "Y").astype(int)
df["is_long_weekend"]      = (df["is_long_weekend"] == "Y").astype(int)

# Season dummies — winter dropped as reference
season_dummies = pd.get_dummies(df["season"], prefix="season", drop_first=False).astype(int)
season_dummies = season_dummies.drop(columns=["season_winter"])
df = pd.concat([df.drop(columns=["season"]), season_dummies], axis=1)

print(f"Shape   : {df.shape}")
print(f"Range   : {df['week_date'].min().date()} → {df['week_date'].max().date()}")
print(f"Missing : {df.isna().sum().sum()}")

In [0]:
CPX_NAVY  = "#002B5C"
CPX_RED   = "#E31837"
CPX_GOLD  = "#F5A800"
CPX_LGREY = "#F2F2F2"
CPX_DGREY = "#4A4A4A"

import os
import matplotlib.patheffects as pe

SAVE_PATH = "/Volumes/data_exploration_hub/marketing_hub/model_data/Final_Data/MMM_EXH_Outcomes/"
os.makedirs(SAVE_PATH, exist_ok=True)

def save_fig(fig, filename):
    """Save figure with Cineplex watermark."""
    fig.text(
        0.99, 0.01,
        f"Cineplex Marketing Analytics  |  EXH MMM Phase 2 Iter 1  |  "
        f"{pd.Timestamp.now().strftime('%Y-%m-%d %H:%M')}",
        ha="right", va="bottom",
        fontsize=6, color=CPX_DGREY, fontstyle="italic",
        transform=fig.transFigure,
    )
    fpath = os.path.join(SAVE_PATH, filename)
    fig.savefig(fpath, dpi=150, bbox_inches="tight", facecolor="white")
    print(f"Saved → {fpath}")

print("✓ Brand colors and save_fig defined")
print(f"  CPX_NAVY  : {CPX_NAVY}")
print(f"  CPX_RED   : {CPX_RED}")
print(f"  CPX_GOLD  : {CPX_GOLD}")
print(f"  Save path : {SAVE_PATH}")

In [0]:
# ── CELL 4: Feature selection, split, and scaling ─────────────────────────────

date_col   = "week_date"
target_col = "total_weekly_attendance"

# Year dummies and trend — add to df before splits
df["is_2024"]    = (df["week_date"].dt.year == 2024).astype(int)
df["is_2025"]    = (df["week_date"].dt.year == 2025).astype(int)
df["week_index"] = (df["week_date"] - df["week_date"].min()).dt.days / 7

# Rebuild control_cols after season encoding
control_cols = df.columns[3:44].tolist()
control_cols = [c for c in control_cols if c != "season"] + season_dummies.columns.tolist()

# 26 features dropped in Phase 1 (multicollinearity + low VIF signal)
DROP_FEATURES = [
    "national_max_temp", "national_min_temp", "national_temp_anomaly",
    "action_intensity", "adaptation_intensity", "adventure_intensity",
    "animated_intensity", "biopic_intensity", "comedy_intensity",
    "crime_intensity", "documentary_intensity", "drama_intensity",
    "family_intensity", "fantasy_intensity", "foreign_intensity",
    "horror_intensity", "musical_intensity", "mystery_intensity",
    "period_intensity", "political_intensity", "romance_intensity",
    "romanticcomedy_intensity", "scifi_intensity", "sequel_intensity",
    "suspense_intensity", "thriller_intensity",
]

baseline_features = [f for f in control_cols if f not in DROP_FEATURES]
baseline_features += ["is_2024", "is_2025", "week_index"]
assert len(baseline_features) == 22, f"Expected 22, got {len(baseline_features)}"

# Split
n       = len(df)
n_test  = 15
n_val   = 26
n_train = n - n_val - n_test

train_df = df.iloc[:n_train].copy().reset_index(drop=True)
val_df   = df.iloc[n_train : n_train + n_val].copy().reset_index(drop=True)
test_df  = df.iloc[n_train + n_val :].copy().reset_index(drop=True)

# Scale — fit only on train
scaler_X = StandardScaler()
scaler_y = StandardScaler()

X_train = scaler_X.fit_transform(train_df[baseline_features])
X_val   = scaler_X.transform(val_df[baseline_features])

y_train = scaler_y.fit_transform(train_df[[target_col]]).flatten()
y_val   = scaler_y.transform(val_df[[target_col]]).flatten()

print(f"Train : {X_train.shape}  ({train_df['week_date'].min().date()} → {train_df['week_date'].max().date()})")
print(f"Val   : {X_val.shape}  ({val_df['week_date'].min().date()} → {val_df['week_date'].max().date()})")
print(f"Test  : held out — not touched until final validation")

In [0]:
all_cols    = df.columns.tolist()
target_idx  = all_cols.index(target_col)
media_start = all_cols.index("Awareness_Cineclub_Digital")
media_cols  = all_cols[media_start:target_idx]

print(f"Media columns: {len(media_cols)}")


def parse_media_col(col):
    parts   = col.split("_")
    funnel  = parts[0].capitalize()
    channel = parts[-1]
    lob     = "_".join(parts[1:-1])
    return funnel, lob, channel


media_meta = pd.DataFrame(
    [parse_media_col(c) + (c,) for c in media_cols],
    columns=["funnel", "lob", "channel", "col_name"],
)

print("Funnels  :", sorted(media_meta["funnel"].unique()))
print("LOBs     :", sorted(media_meta["lob"].unique()))
print("Channels :", sorted(media_meta["channel"].unique()))
print("\nCols per funnel:\n", media_meta["funnel"].value_counts())

# Aggregate spend to funnel level on the full df
FUNNELS = ["Awareness", "Consideration", "Conversion"]
for funnel in FUNNELS:
    fcols = media_meta.loc[media_meta["funnel"] == funnel, "col_name"].tolist()
    df[f"spend_{funnel}"] = df[fcols].sum(axis=1)

# Propagate to splits
for split in [train_df, val_df, test_df]:
    for funnel in FUNNELS:
        split[f"spend_{funnel}"] = df.loc[
            df["week_date"].isin(split["week_date"]), f"spend_{funnel}"
        ].values

In [0]:
spend_train_max = {}
for funnel in FUNNELS:
    spend_train_max[funnel] = float(train_df[f"spend_{funnel}"].max())
    print(f"{funnel:<15} train max: ${spend_train_max[funnel]:>12,.0f}")

spend_scaled = {}
for funnel in FUNNELS:
    m = spend_train_max[funnel]
    train_vals    = train_df[f"spend_{funnel}"].values.astype(np.float64)
    val_vals      = val_df[f"spend_{funnel}"].values.astype(np.float64)
    trainval_vals = np.concatenate([train_vals, val_vals])

    spend_scaled[funnel] = {
        "train"    : train_vals    / m,
        "trainval" : trainval_vals / m,
    }
    print(f"{funnel:<15} scaled train range: [{spend_scaled[funnel]['train'].min():.3f}, {spend_scaled[funnel]['train'].max():.3f}]")


In [0]:

def geometric_adstock(x, alpha):
    """
    Recursive geometric adstock. Equivalent to infinite-lag convolution
    with weights [1, alpha, alpha^2, ...]. Effective window controlled by alpha.

    x     : np.ndarray shape (T,) — pre-scaled spend (constant, not an RV)
    alpha : pytensor scalar       — decay rate, learned inside the model

    Returns pytensor vector shape (T,)
    """
    x_t = pt.as_tensor_variable(x.astype(np.float64))

    def step(x_curr, adstock_prev, alpha):
        return x_curr + alpha * adstock_prev

    adstocked, _ = pytensor.scan(
        fn=step,
        sequences=[x_t],
        outputs_info=[pt.as_tensor_variable(np.float64(0.0))],
        non_sequences=[alpha],
    )
    return adstocked


def logistic_saturation(x, lam):
    """
    Logistic saturation: f(x) = 1 - exp(-lam * x)
    Maps [0, inf) → [0, 1). lam controls speed of diminishing returns.

    x   : pytensor vector
    lam : pytensor scalar — saturation rate, learned inside the model
    """
    return 1 - pt.exp(-lam * x)

# FUNNEL LEVEL MODEL

In [0]:
def hill_saturation(x, k, alpha_s):
    x_safe = pt.maximum(x, 1e-6)
    return (x_safe ** alpha_s) / (x_safe ** alpha_s + k ** alpha_s)

def logistic_saturation(x, lam):
    return 1 - pt.exp(-lam * x)

FUNNELS = ["Awareness", "Consideration", "Conversion"]

ALPHA_PRIORS = {
    "Awareness"    : {"alpha": 7, "beta": 2},
    "Consideration": {"alpha": 3, "beta": 3},
    "Conversion"   : {"alpha": 2, "beta": 5},
}

with pm.Model() as mmm_iter1_mixed:

    intercept = pm.Normal("intercept", mu=0, sigma=1)
    beta      = pm.Normal("beta", mu=0, sigma=1, shape=X_train.shape[1])
    sigma     = pm.HalfNormal("sigma", sigma=1)

    mu = intercept + pm.math.dot(X_train, beta)

    for funnel in FUNNELS:
        ap      = ALPHA_PRIORS[funnel]
        alpha_f = pm.Beta(f"alpha_{funnel}", alpha=ap["alpha"], beta=ap["beta"])
        gamma_f = pm.HalfNormal(f"gamma_{funnel}", sigma=0.5)

        adstocked = geometric_adstock(spend_scaled[funnel]["train"], alpha_f)

        if funnel == "Consideration":
            # Hill — threshold effect justified for mid-funnel
            k_f       = pm.HalfNormal("k_Consideration",       sigma=0.5)
            alpha_s_f = pm.HalfNormal("alpha_s_Consideration", sigma=1.5)
            saturated = hill_saturation(adstocked, k_f, alpha_s_f)
        else:
            # Logistic — diminishing returns from first dollar
            lam_f     = pm.HalfNormal(f"lam_{funnel}", sigma=1)
            saturated = logistic_saturation(adstocked, lam_f)

        mu = mu + gamma_f * saturated

    y_obs = pm.Normal("y_obs", mu=mu, sigma=sigma, observed=y_train)

    trace_iter1_mixed = pmjax.sample_numpyro_nuts(
        draws=2000, tune=2000,
        target_accept=0.99,
        chains=4,
        random_seed=42,
    )

In [0]:
MEDIA_PARAMS_MIXED = [
    "alpha_Awareness",   "alpha_Consideration",   "alpha_Conversion",
    "lam_Awareness",     "lam_Conversion",
    "k_Consideration",   "alpha_s_Consideration",
    "gamma_Awareness",   "gamma_Consideration",   "gamma_Conversion",
]
ALL_PARAMS_MIXED = ["intercept", "beta", "sigma"] + MEDIA_PARAMS_MIXED

summary_mixed = az.summary(trace_iter1_mixed, var_names=ALL_PARAMS_MIXED)
summary_mixed["r_hat"] = pd.to_numeric(summary_mixed["r_hat"], errors="coerce")

max_rhat    = float(summary_mixed["r_hat"].max())
min_ess     = int(summary_mixed["ess_bulk"].min())
divergences = int(trace_iter1_mixed.sample_stats.diverging.sum().item())

print("=" * 55)
print("CONVERGENCE — Iteration 1 (Mixed saturation)")
print("=" * 55)
print(f"Max R-hat   : {max_rhat:.4f}   (target < 1.01)")
print(f"Min ESS     : {min_ess}        (target > 400)")
print(f"Divergences : {divergences}    (target 0)")

print("\n--- Media parameter posteriors ---")
print(summary_mixed.loc[
    summary_mixed.index.intersection(MEDIA_PARAMS_MIXED),
    ['mean', 'sd', 'eti89_lb', 'eti89_ub', 'ess_bulk', 'r_hat']
].to_string())


In [0]:
post = trace_iter1_mixed.posterior

intercept_mean = float(post["intercept"].values.ravel().mean())
beta_mean      = post["beta"].values.reshape(-1, len(baseline_features)).mean(axis=0)

X_trainval  = np.vstack([X_train, X_val])
mu_controls = intercept_mean + X_trainval @ beta_mean

mu_media      = np.zeros(n_train + n_val)
media_contrib = {}

for funnel in FUNNELS:
    alpha_mean = float(post[f"alpha_{funnel}"].values.ravel().mean())
    gamma_mean = float(post[f"gamma_{funnel}"].values.ravel().mean())

    x_spend = spend_scaled[funnel]["trainval"]

    adstocked_np    = np.zeros(len(x_spend))
    adstocked_np[0] = x_spend[0]
    for t in range(1, len(x_spend)):
        adstocked_np[t] = x_spend[t] + alpha_mean * adstocked_np[t - 1]

    if funnel == "Consideration":
        k_mean       = float(post["k_Consideration"].values.ravel().mean())
        alpha_s_mean = float(post["alpha_s_Consideration"].values.ravel().mean())
        x_safe       = np.maximum(adstocked_np, 1e-6)
        saturated_np = (x_safe ** alpha_s_mean) / (
                        x_safe ** alpha_s_mean + k_mean ** alpha_s_mean)
    else:
        lam_mean     = float(post[f"lam_{funnel}"].values.ravel().mean())
        saturated_np = 1 - np.exp(-lam_mean * adstocked_np)

    contrib_scaled        = gamma_mean * saturated_np
    mu_media             += contrib_scaled
    media_contrib[funnel] = contrib_scaled * scaler_y.scale_[0]

pred_trainval  = scaler_y.inverse_transform(
    (mu_controls + mu_media).reshape(-1, 1)
).ravel()

pred_train_v   = pred_trainval[:n_train]
pred_val_v     = pred_trainval[n_train:]
actual_train_v = train_df[target_col].values
actual_val_v   = val_df[target_col].values

mape_train = mean_absolute_percentage_error(actual_train_v, pred_train_v) * 100
mape_val   = mean_absolute_percentage_error(actual_val_v,   pred_val_v)   * 100
r2_train   = r2_score(actual_train_v, pred_train_v)
r2_val     = r2_score(actual_val_v,   pred_val_v)

# ── Store for comparison table in Cell 20 ────────────────────────────────────
I1_MIX_TRAIN_MAPE = mape_train
I1_MIX_VAL_MAPE   = mape_val
I1_MIX_TRAIN_R2   = r2_train
I1_MIX_VAL_R2     = r2_val

resid_train = actual_train_v - pred_train_v
resid_val   = actual_val_v   - pred_val_v

print("=" * 55)
print("ITERATION 1 MIXED — Metrics")
print("=" * 55)
print(f"{'':22} {'Train MAPE':>11} {'Val MAPE':>10} {'Train R²':>10} {'Val R²':>8}")
print(f"{'Mixed (Iter 1)':22} {mape_train:>10.2f}% {mape_val:>9.2f}% "
      f"{r2_train:>10.3f} {r2_val:>8.3f}")

print("\n--- Residual summary ---")
print(f"{'':20} {'Train':>15} {'Val':>15}")
print(f"{'Mean':20} {resid_train.mean():>15,.0f} {resid_val.mean():>15,.0f}")
print(f"{'Std':20} {resid_train.std():>15,.0f} {resid_val.std():>15,.0f}")
print(f"{'Median |err|':20} {np.median(np.abs(resid_train)):>15,.0f} "
      f"{np.median(np.abs(resid_val)):>15,.0f}")

In [0]:
# #FUNNELS = ["Awareness", "Consideration", "Conversion"]

# df["spend_Awareness"]     = df[[c for c in media_meta.loc[media_meta["funnel"]=="Awareness",    "col_name"]]].sum(axis=1)
# df["spend_Consideration"] = df[[c for c in media_meta.loc[media_meta["funnel"]=="Consideration","col_name"]]].sum(axis=1)
# df["spend_Conversion"]    = df[[c for c in media_meta.loc[media_meta["funnel"]=="Conversion",   "col_name"]]].sum(axis=1)

# for split in [train_df, val_df, test_df]:
#     for funnel in FUNNELS:
#         split[f"spend_{funnel}"] = df.loc[
#             df["week_date"].isin(split["week_date"]), f"spend_{funnel}"
#         ].values

# ALPHA_PRIORS = {
#     "Awareness"    : {"alpha": 5, "beta": 2},
#     "Consideration": {"alpha": 2, "beta": 2},
#     "Conversion"   : {"alpha": 2, "beta": 3},
# }

# spend_train_max = {}
# for funnel in FUNNELS:
#     spend_train_max[funnel] = float(train_df[f"spend_{funnel}"].max())
#     print(f"{funnel:<15} train max: ${spend_train_max[funnel]:>12,.0f}")

# spend_scaled = {}
# for funnel in FUNNELS:
#     m             = spend_train_max[funnel]
#     train_vals    = train_df[f"spend_{funnel}"].values.astype(np.float64)
#     val_vals      = val_df[f"spend_{funnel}"].values.astype(np.float64)
#     trainval_vals = np.concatenate([train_vals, val_vals])
#     spend_scaled[funnel] = {
#         "train"    : train_vals    / m,
#         "trainval" : trainval_vals / m,
#     }
#     print(f"{funnel:<15} scaled train range: [{spend_scaled[funnel]['train'].min():.3f}, {spend_scaled[funnel]['train'].max():.3f}]")

# with pm.Model() as mmm_iter1:

#     intercept = pm.Normal("intercept", mu=0, sigma=1)
#     beta      = pm.Normal("beta", mu=0, sigma=1, shape=X_train.shape[1])
#     sigma     = pm.HalfNormal("sigma", sigma=1)

#     mu = intercept + pm.math.dot(X_train, beta)

#     for funnel in FUNNELS:
#         ap      = ALPHA_PRIORS[funnel]
#         alpha_f = pm.Beta(f"alpha_{funnel}",  alpha=ap["alpha"], beta=ap["beta"])
#         lam_f   = pm.HalfNormal(f"lam_{funnel}",   sigma=1)
#         gamma_f = pm.HalfNormal(f"gamma_{funnel}", sigma=0.5)

#         adstocked = geometric_adstock(spend_scaled[funnel]["train"], alpha_f)
#         saturated = logistic_saturation(adstocked, lam_f)

#         mu = mu + gamma_f * saturated

#     y_obs = pm.Normal("y_obs", mu=mu, sigma=sigma, observed=y_train)

#     trace_iter1 = pmjax.sample_numpyro_nuts(
#         draws=2000, tune=2000,
#         target_accept=0.95,
#         chains=4,
#         random_seed=42,
#     )
# 
# 
# 
# 
#  I2_TRAIN_MAPE = 19.74
# I2_VAL_MAPE   = 22.99
# I2_TRAIN_R2   = 0.633
# I2_VAL_R2     = 0.476

# intercept_mean = float(post["intercept"].values.ravel().mean())
# beta_mean      = post["beta"].values.reshape(-1, len(baseline_features)).mean(axis=0)

# X_trainval  = np.vstack([X_train, X_val])
# mu_controls = intercept_mean + X_trainval @ beta_mean

# mu_media      = np.zeros(n_train + n_val)
# media_contrib = {}

# post = trace_iter1_hill.posterior

# # Replace lam_mean with Hill parameters
# for funnel in FUNNELS:
#     alpha_mean   = float(post[f"alpha_{funnel}"].values.ravel().mean())
#     k_mean       = float(post[f"k_{funnel}"].values.ravel().mean())
#     alpha_s_mean = float(post[f"alpha_s_{funnel}"].values.ravel().mean())
#     gamma_mean   = float(post[f"gamma_{funnel}"].values.ravel().mean())

#     x_spend = spend_scaled[funnel]["trainval"]

#     adstocked_np    = np.zeros(len(x_spend))
#     adstocked_np[0] = x_spend[0]
#     for t in range(1, len(x_spend)):
#         adstocked_np[t] = x_spend[t] + alpha_mean * adstocked_np[t - 1]

#     x_safe       = np.maximum(adstocked_np, 1e-6)
#     saturated_np = (x_safe ** alpha_s_mean) / (x_safe ** alpha_s_mean + k_mean ** alpha_s_mean)

#     contrib_scaled = gamma_mean * saturated_np
#     mu_media      += contrib_scaled
#     media_contrib[funnel] = contrib_scaled * scaler_y.scale_[0]

# pred_trainval  = scaler_y.inverse_transform(
#     (mu_controls + mu_media).reshape(-1, 1)
# ).ravel()

# pred_train_v   = pred_trainval[:n_train]
# pred_val_v     = pred_trainval[n_train:]
# actual_train_v = train_df[target_col].values
# actual_val_v   = val_df[target_col].values

# mape_train = mean_absolute_percentage_error(actual_train_v, pred_train_v) * 100
# mape_val   = mean_absolute_percentage_error(actual_val_v,   pred_val_v)   * 100
# r2_train   = r2_score(actual_train_v, pred_train_v)
# r2_val     = r2_score(actual_val_v,   pred_val_v)

# print("=" * 68)
# print("ITERATION 1 vs PHASE 1 BASELINE")
# print("=" * 68)
# print(f"{'':22} {'Train MAPE':>11} {'Val MAPE':>10} {'Train R²':>10} {'Val R²':>8}")
# print(f"{'Phase 1 (baseline)':22} {P1_TRAIN_MAPE:>10.2f}% {P1_VAL_MAPE:>9.2f}% {P1_TRAIN_R2:>10.3f} {P1_VAL_R2:>8.3f}")
# print(f"{'Iteration 1':22} {mape_train:>10.2f}% {mape_val:>9.2f}% {r2_train:>10.3f} {r2_val:>8.3f}")

# delta_mape = mape_val - P1_VAL_MAPE
# delta_r2   = r2_val   - P1_VAL_R2
# print(f"\nVal MAPE delta : {delta_mape:+.2f} pts  {'↓ BETTER' if delta_mape < 0 else '↑ WORSE'}")
# print(f"Val R²   delta : {delta_r2:+.3f}      {'↑ BETTER' if delta_r2 > 0 else '↓ WORSE'}")

# resid_train = actual_train_v - pred_train_v
# resid_val   = actual_val_v   - pred_val_v

# print("\n--- Residual summary ---")
# print(f"{'':20} {'Train':>15} {'Val':>15}")
# print(f"{'Mean':20} {resid_train.mean():>15,.0f} {resid_val.mean():>15,.0f}")
# print(f"{'Std':20} {resid_train.std():>15,.0f} {resid_val.std():>15,.0f}")
# print(f"{'Median |err|':20} {np.median(np.abs(resid_train)):>15,.0f} {np.median(np.abs(resid_val)):>15,.0f}")

# SAVE_PATH = "/Volumes/data_exploration_hub/marketing_hub/model_data/Final_Data/Data_Exploration/"
# RUN_TS    = datetime.datetime.now().strftime("%Y-%m-%d %H:%M")

# CPX_NAVY  = "#002B5C"
# CPX_RED   = "#E31837"
# CPX_GOLD  = "#F5A800"
# CPX_LGREY = "#F2F2F2"
# CPX_DGREY = "#4A4A4A"
# CPX_WHITE = "#FFFFFF"

# FUNNEL_COLORS = {                                                        # ← fixed
#     "UpperFunnel": CPX_NAVY,
#     "Conversion" : CPX_GOLD,
# }

# mpl.rcParams.update({
#     "figure.facecolor": CPX_WHITE, "axes.facecolor":   CPX_WHITE,
#     "axes.edgecolor":   CPX_DGREY, "axes.labelcolor":  CPX_DGREY,
#     "axes.grid":        True,      "grid.color":       CPX_LGREY,
#     "grid.linewidth":   0.8,       "xtick.color":      CPX_DGREY,
#     "ytick.color":      CPX_DGREY, "text.color":       CPX_DGREY,
#     "font.family":      "sans-serif", "font.size":     10,
#     "legend.framealpha": 0.9,     "legend.edgecolor": CPX_LGREY,
# })


# def add_watermark(fig):
#     fig.text(
#         0.99, 0.01,
#         f"Cineplex Marketing Analytics  |  EXH MMM Phase 2 Iter 1  |  {RUN_TS}",
#         ha="right", va="bottom", fontsize=7, color=CPX_DGREY, alpha=0.6, fontstyle="italic",
#     )


# def save_fig(fig, filename):
#     add_watermark(fig)
#     fig.savefig(SAVE_PATH + filename, dpi=300, bbox_inches="tight")
#     print(f"Saved: {filename}")


# df_trainval = pd.concat([train_df, val_df]).copy().reset_index(drop=True)
# df_trainval["segment"]       = ["train"] * n_train + ["val"] * n_val
# df_trainval["pred"]          = pred_trainval
# df_trainval["resid"]         = df_trainval[target_col].values - pred_trainval
# df_trainval["controls_pred"] = scaler_y.inverse_transform(mu_controls.reshape(-1, 1)).ravel()

# for funnel in FUNNELS:
#     df_trainval[f"media_{funnel}"] = media_contrib[funnel]

# df_trainval["total_media"] = sum(media_contrib[f] for f in FUNNELS)

In [0]:
# # ── CELL 19: Business — ROAS summary table ────────────────────────────────────
# # ROAS = incremental attendance attributed / total spend (CAD)
# # Uses train+val period

# roas_rows = []
# for funnel in FUNNELS:
#     total_spend_cad   = (
#         train_df[f"spend_{funnel}"].sum() + val_df[f"spend_{funnel}"].sum()
#     )
#     total_contrib_att = df_trainval[f"media_{funnel}"].sum()
#     roas              = total_contrib_att / total_spend_cad if total_spend_cad > 0 else np.nan

#     roas_rows.append({
#         "Funnel"                  : funnel,
#         "Total Spend (CAD)"       : total_spend_cad,
#         "Attributed Attendance"   : total_contrib_att,
#         "ROAS (att / $CAD)"       : roas,
#         "Spend Share %"           : None,  # filled below
#         "Contribution Share %"    : None,
#     })

# roas_df = pd.DataFrame(roas_rows)
# total_spend = roas_df["Total Spend (CAD)"].sum()
# total_att   = roas_df["Attributed Attendance"].sum()
# roas_df["Spend Share %"]        = (roas_df["Total Spend (CAD)"]     / total_spend * 100).round(1)
# roas_df["Contribution Share %"] = (roas_df["Attributed Attendance"] / total_att   * 100).round(1)

# # Format for display
# roas_display = roas_df.copy()
# roas_display["Total Spend (CAD)"]     = roas_display["Total Spend (CAD)"].map("${:,.0f}".format)
# roas_display["Attributed Attendance"] = roas_display["Attributed Attendance"].map("{:,.0f}".format)
# roas_display["ROAS (att / $CAD)"]     = roas_display["ROAS (att / $CAD)"].map("{:.4f}".format)

# print("=" * 75)
# print("ROAS SUMMARY — Iteration 1  (train + val | posterior mean estimates)")
# print("=" * 75)
# print(roas_display.to_string(index=False))
# print(f"\nTotal attributed media attendance : {total_att:,.0f}")
# print(f"Total media spend (CAD)           : ${total_spend:,.0f}")
# print(f"Blended ROAS                      : {total_att/total_spend:.4f} att/$")
# print("\nNote: ROAS at this stage is approximate (posterior mean, not full posterior).")
# print("Full posterior ROAS uncertainty will be quantified in later iterations.")

# LOB LEVEL MODEL

In [0]:
# ── CELL 20: Iteration 2 — Funnel x LOB spend aggregation and scaling ─────────

# Build all funnel x LOB combinations that exist in media_meta
fl_combos = (
    media_meta.groupby(["funnel", "lob"])["col_name"]
    .apply(list)
    .reset_index()
)
fl_combos["fl_key"] = fl_combos["funnel"] + "__" + fl_combos["lob"]

print(f"Funnel x LOB combinations: {len(fl_combos)}")
print()

# Aggregate spend per combination on full df
for _, row in fl_combos.iterrows():
    df[row["fl_key"]] = df[row["col_name"]].sum(axis=1)

# Propagate to splits
for split in [train_df, val_df]:
    for _, row in fl_combos.iterrows():
        split[row["fl_key"]] = df.loc[
            df["week_date"].isin(split["week_date"]), row["fl_key"]
        ].values

# Zero-week audit — flag sparse combinations upfront
print(f"{'Funnel x LOB':<50} {'Zero weeks (train)':>20} {'% zero':>8}")
print("─" * 80)
sparse_keys = []
for _, row in fl_combos.iterrows():
    key      = row["fl_key"]
    zeros    = (train_df[key] == 0).sum()
    pct_zero = zeros / len(train_df) * 100
    flag     = " ← SPARSE" if pct_zero > 60 else ""
    print(f"{key:<50} {zeros:>20}  {pct_zero:>6.1f}%{flag}")
    if pct_zero > 60:
        sparse_keys.append(key)

print(f"\nSparse combinations (>60% zero weeks): {len(sparse_keys)}")

# Scale by train-period max per combination
fl_spend_train_max = {}
fl_spend_scaled    = {}

for _, row in fl_combos.iterrows():
    key = row["fl_key"]
    m   = float(train_df[key].max())

    # Avoid divide-by-zero for all-zero combinations (shouldn't exist but safety check)
    if m == 0:
        print(f"WARNING: {key} has zero max spend in train — skipping")
        continue

    fl_spend_train_max[key] = m
    train_vals    = train_df[key].values.astype(np.float64)
    val_vals      = val_df[key].values.astype(np.float64)

    fl_spend_scaled[key] = {
        "train"    : train_vals    / m,
        "trainval" : np.concatenate([train_vals, val_vals]) / m,
    }

print(f"\nScaled {len(fl_spend_scaled)} combinations.")

In [0]:

LOB_GROUPS = {
    "Premium_Brand" : [
        "Premium Experiences - AVX",
        "Premium Experiences - VIP",
        "F&B",
        "EXH - Brand",
    ],
    "Film_Marketing": [
        "Premium Experiences - IMAX",
        "Film Marketing - Inform",
        "Performance - Convert & Upgrade",
    ],
    "Prepaid"       : ["Prepaid - B2B", "Prepaid - B2C"],
    "Cineclub"      : ["Cineclub"],
    "GE"            : ["G&E"],
}

FUNNELS_ITER2 = ["Awareness", "Consideration", "Conversion"]

# ── Step 1: Build funnel x LOB group columns on df ───────────────────────────
for funnel in FUNNELS_ITER2:
    for lob_group, lobs in LOB_GROUPS.items():
        col_name = f"{funnel}__{lob_group}"
        matching = media_meta.loc[
            (media_meta["funnel"] == funnel) &
            (media_meta["lob"].isin(lobs)),
            "col_name"
        ].tolist()
        df[col_name] = df[matching].sum(axis=1) if matching else 0.0

# ── Step 2: Build 10-term collapsed structure ─────────────────────────────────
ITER2_COMBOS = {
    "Awareness__Premium_Brand"     : ["Awareness__Premium_Brand"],
    "Awareness__Other"             : ["Awareness__Cineclub", "Awareness__Film_Marketing"],
    "Consideration__Premium_Brand" : ["Consideration__Premium_Brand"],
    "Consideration__GE"            : ["Consideration__GE"],
    "Consideration__Prepaid"       : ["Consideration__Prepaid"],
    "Consideration__Film_Marketing": ["Consideration__Film_Marketing"],
    "Conversion__Film_Marketing"   : ["Conversion__Film_Marketing"],
    "Conversion__Premium_Brand"    : ["Conversion__Premium_Brand"],
    "Conversion__Cineclub"         : ["Conversion__Cineclub"],
    "Conversion__Other"            : ["Conversion__Prepaid", "Conversion__GE"],
}

for combo_name, source_cols in ITER2_COMBOS.items():
    df[combo_name] = df[source_cols].sum(axis=1)

# ── Step 3: Propagate to splits ───────────────────────────────────────────────
for split in [train_df, val_df, test_df]:
    for combo_name in ITER2_COMBOS:
        split[combo_name] = df.loc[
            df["week_date"].isin(split["week_date"]), combo_name
        ].values

# ── Step 4: Drop sparse / low-spend combinations ──────────────────────────────
DROP_SPARSE = [
    "Awareness__Other",           # 79.8% zero, $3,758 max
    "Consideration__Film_Marketing",  # 70.2% zero
    "Conversion__Other",          # 94.4% zero, $7,556 max
]
ACTIVE_COMBOS = [c for c in ITER2_COMBOS if c not in DROP_SPARSE]

# ── Step 5: Scale — max normalise on train ────────────────────────────────────
spend_scaled_iter2    = {}
spend_train_max_iter2 = {}

print(f"{'Combination':<35} {'Train max':>14}  {'Scaled range'}")
print("─" * 70)
for combo_name in ACTIVE_COMBOS:
    m          = float(train_df[combo_name].max())
    train_vals = train_df[combo_name].values.astype(np.float64)
    val_vals   = val_df[combo_name].values.astype(np.float64)
    trainval   = np.concatenate([train_vals, val_vals])

    spend_train_max_iter2[combo_name] = m
    spend_scaled_iter2[combo_name]    = {
        "train"    : train_vals / m,
        "trainval" : trainval   / m,
    }
    print(f"{combo_name:<35} ${m:>12,.0f}  "
          f"[{train_vals.min()/m:.3f}, {train_vals.max()/m:.3f}]")
    
    # ── FUNNEL_COMBOS: maps funnel → list of active combos ───────────────────────
FUNNEL_COMBOS = {
    funnel: [c for c in ACTIVE_COMBOS if c.startswith(funnel)]
    for funnel in ["Awareness", "Consideration", "Conversion"]
}

# Also define LOB_GROUP_MAP for original LOB attribution downstream
LOB_GROUP_MAP = {
    lob_group: lobs
    for lob_group, lobs in LOB_GROUPS.items()
}

print("\nFUNNEL_COMBOS:")
for funnel, combos in FUNNEL_COMBOS.items():
    print(f"  {funnel:<15} : {combos}")


In [0]:
# ── CELL: Iteration 2 — Mixed Hierarchical Model ─────────────────────────────
#
# Structure:
#   Awareness LOBs    → Logistic + hierarchical pooling
#   Consideration LOBs → Hill    + hierarchical pooling (k and alpha_s pooled)
#   Conversion LOBs   → Logistic + hierarchical pooling

def hill_saturation(x, k, alpha_s):
    x_safe = pt.maximum(x, 1e-6)
    return (x_safe ** alpha_s) / (x_safe ** alpha_s + k ** alpha_s)

def logistic_saturation(x, lam):
    return 1 - pt.exp(-lam * x)

ALPHA_MU_PRIOR = {
    "Awareness"    : {"alpha": 7, "beta": 2},   # mean ~0.78
    "Consideration": {"alpha": 3, "beta": 3},   # mean ~0.50 — tightened
    "Conversion"   : {"alpha": 2, "beta": 5},   # mean ~0.29
}

MU_GAMMA_PRIOR = {
    "Awareness"    : 0.5,
    "Consideration": 0.3,
    "Conversion"   : 0.6,
}

with pm.Model() as mmm_iter2_mixed_hier:

    intercept = pm.Normal("intercept", mu=0, sigma=1)
    beta      = pm.Normal("beta", mu=0, sigma=1, shape=X_train.shape[1])
    sigma     = pm.HalfNormal("sigma", sigma=1)

    mu = intercept + pm.math.dot(X_train, beta)

    for funnel, combos in FUNNEL_COMBOS.items():
        ap = ALPHA_MU_PRIOR[funnel]

        # ── Funnel-level hyperpriors — shared across all ──────────────────────
        mu_alpha_f    = pm.Beta(
            f"mu_alpha_{funnel}",
            alpha=ap["alpha"], beta=ap["beta"]
        )
        kappa_alpha_f = pm.HalfNormal(f"kappa_alpha_{funnel}", sigma=2)
        mu_gamma_f    = pm.HalfNormal(f"mu_gamma_{funnel}",
                                       sigma=MU_GAMMA_PRIOR[funnel])

        # ── Consideration only: pool Hill parameters ──────────────────────────
        if funnel == "Consideration":
            mu_k_f       = pm.HalfNormal("mu_k_Consideration",       sigma=0.5)
            mu_alpha_s_f = pm.HalfNormal("mu_alpha_s_Consideration", sigma=1.5)

        for combo in combos:

            # ── Alpha: pooled decay ───────────────────────────────────────────
            alpha_c = pm.Beta(
                f"alpha_{combo}",
                alpha = mu_alpha_f    * kappa_alpha_f,
                beta  = (1 - mu_alpha_f) * kappa_alpha_f,
            )

            # ── Gamma: pooled effect size ─────────────────────────────────────
            gamma_c = pm.HalfNormal(f"gamma_{combo}", sigma=mu_gamma_f)

            adstocked = geometric_adstock(
                spend_scaled_iter2[combo]["train"], alpha_c
            )

            if funnel == "Consideration":
                # Hill — pooled k and alpha_s
                k_c       = pm.HalfNormal(f"k_{combo}",       sigma=mu_k_f)
                alpha_s_c = pm.HalfNormal(f"alpha_s_{combo}", sigma=mu_alpha_s_f)
                saturated = hill_saturation(adstocked, k_c, alpha_s_c)
            else:
                # Logistic — pooled lam
                lam_c     = pm.HalfNormal(f"lam_{combo}", sigma=1)
                saturated = logistic_saturation(adstocked, lam_c)

            mu = mu + gamma_c * saturated

    y_obs = pm.Normal("y_obs", mu=mu, sigma=sigma, observed=y_train)

    trace_iter2_mixed_hier = pmjax.sample_numpyro_nuts(
        draws=2000, tune=2000,
        target_accept=0.99,
        chains=4,
        random_seed=42,
    )

In [0]:
HYPER_PARAMS_MH = (
    [f"mu_alpha_{f}"    for f in FUNNEL_COMBOS] +
    [f"kappa_alpha_{f}" for f in FUNNEL_COMBOS] +
    [f"mu_gamma_{f}"    for f in FUNNEL_COMBOS] +
    ["mu_k_Consideration", "mu_alpha_s_Consideration"]
)

LOB_PARAMS_MH = []
for funnel, combos in FUNNEL_COMBOS.items():
    LOB_PARAMS_MH += [f"alpha_{c}" for c in combos]
    LOB_PARAMS_MH += [f"gamma_{c}" for c in combos]
    if funnel == "Consideration":
        LOB_PARAMS_MH += [f"k_{c}"       for c in combos]
        LOB_PARAMS_MH += [f"alpha_s_{c}" for c in combos]
    else:
        LOB_PARAMS_MH += [f"lam_{c}" for c in combos]

ALL_PARAMS_MH = ["intercept", "beta", "sigma"] + HYPER_PARAMS_MH + LOB_PARAMS_MH

summary_mh = az.summary(trace_iter2_mixed_hier, var_names=ALL_PARAMS_MH)
summary_mh["r_hat"] = pd.to_numeric(summary_mh["r_hat"], errors="coerce")

max_rhat    = float(summary_mh["r_hat"].max())
min_ess     = int(summary_mh["ess_bulk"].min())
divergences = int(trace_iter2_mixed_hier.sample_stats.diverging.sum().item())

print("=" * 55)
print("CONVERGENCE — Iteration 2 (Mixed Hierarchical)")
print("=" * 55)
print(f"Max R-hat   : {max_rhat:.4f}   (target < 1.01)")
print(f"Min ESS     : {min_ess}        (target > 400)")
print(f"Divergences : {divergences}    (target 0)")

print("\n--- Hyperparameter posteriors ---")
print(summary_mh.loc[
    summary_mh.index.intersection(HYPER_PARAMS_MH),
    ['mean', 'sd', 'eti89_lb', 'eti89_ub', 'ess_bulk', 'r_hat']
].to_string())

print("\n--- LOB parameter posteriors ---")
print(summary_mh.loc[
    summary_mh.index.intersection(LOB_PARAMS_MH),
    ['mean', 'sd', 'eti89_lb', 'eti89_ub', 'ess_bulk', 'r_hat']
].to_string())

# ── Store Iter 2 Hierarchical metrics for comparison table ────────────────────
# Replace the commented-out hardcoded lines with live assignments
# Run this cell after trace_iter2_hierarchical sampling completes
I2_TRAIN_MAPE = 19.74
I2_VAL_MAPE   = 22.99
I2_TRAIN_R2   = 0.633
I2_VAL_R2     = 0.476

In [0]:
# ── CELL: Iteration 2 Mixed Hierarchical — Metrics + df_trainval build ────────

# ── Prior iteration metrics (hardcoded reference values) ─────────────────────
# Logistic Iter 1
I1_TRAIN_MAPE  = 20.76;  I1_VAL_MAPE  = 23.74
I1_TRAIN_R2    = 0.627;  I1_VAL_R2    = 0.467

# Mixed Iter 1
I1_MIX_TRAIN_MAPE  = 19.97;  I1_MIX_VAL_MAPE  = 23.31
I1_MIX_TRAIN_R2    = 0.635;  I1_MIX_VAL_R2    = 0.460

# Hierarchical Iter 2 (non-mixed)
I2_TRAIN_MAPE  = 19.74;  I2_VAL_MAPE  = 22.99
I2_TRAIN_R2    = 0.633;  I2_VAL_R2    = 0.476

# ── Posterior means ───────────────────────────────────────────────────────────
post = trace_iter2_mixed_hier.posterior

intercept_mean = float(post["intercept"].values.ravel().mean())
beta_mean      = post["beta"].values.reshape(-1, len(baseline_features)).mean(axis=0)

X_trainval  = np.vstack([X_train, X_val])
mu_controls = intercept_mean + X_trainval @ beta_mean

mu_media      = np.zeros(n_train + n_val)
media_contrib = {}

for funnel, combos in FUNNEL_COMBOS.items():
    for combo in combos:
        alpha_mean = float(post[f"alpha_{combo}"].values.ravel().mean())
        gamma_mean = float(post[f"gamma_{combo}"].values.ravel().mean())

        x_spend = spend_scaled_iter2[combo]["trainval"]

        adstocked_np    = np.zeros(len(x_spend))
        adstocked_np[0] = x_spend[0]
        for t in range(1, len(x_spend)):
            adstocked_np[t] = x_spend[t] + alpha_mean * adstocked_np[t - 1]

        if funnel == "Consideration":
            k_mean       = float(post[f"k_{combo}"].values.ravel().mean())
            alpha_s_mean = float(post[f"alpha_s_{combo}"].values.ravel().mean())
            x_safe       = np.maximum(adstocked_np, 1e-6)
            saturated_np = (x_safe ** alpha_s_mean) / (
                           x_safe ** alpha_s_mean + k_mean ** alpha_s_mean)
        else:
            lam_mean     = float(post[f"lam_{combo}"].values.ravel().mean())
            saturated_np = 1 - np.exp(-lam_mean * adstocked_np)

        contrib_scaled        = gamma_mean * saturated_np
        mu_media             += contrib_scaled
        media_contrib[combo]  = contrib_scaled * scaler_y.scale_[0]

# ── Predictions ───────────────────────────────────────────────────────────────
pred_trainval  = scaler_y.inverse_transform(
                     (mu_controls + mu_media).reshape(-1, 1)).ravel()

pred_train_v   = pred_trainval[:n_train]
pred_val_v     = pred_trainval[n_train:]
actual_train_v = train_df[target_col].values
actual_val_v   = val_df[target_col].values

resid_train    = actual_train_v - pred_train_v
resid_val      = actual_val_v   - pred_val_v

# ── Metrics ───────────────────────────────────────────────────────────────────
mape_train = mean_absolute_percentage_error(actual_train_v, pred_train_v) * 100
mape_val   = mean_absolute_percentage_error(actual_val_v,   pred_val_v)   * 100
r2_train   = r2_score(actual_train_v, pred_train_v)
r2_val     = r2_score(actual_val_v,   pred_val_v)

# Store for downstream use
I2_MIX_HIER_TRAIN_MAPE = mape_train;  I2_MIX_HIER_VAL_MAPE = mape_val
I2_MIX_HIER_TRAIN_R2   = r2_train;    I2_MIX_HIER_VAL_R2   = r2_val

# ── Comparison table ──────────────────────────────────────────────────────────
print("=" * 72)
print("MODEL PROGRESSION — FULL COMPARISON")
print("=" * 72)
print(f"{'':28} {'Train MAPE':>11} {'Val MAPE':>10} {'Train R²':>10} {'Val R²':>8}")
print("─" * 72)
print(f"{'Logistic Iter 1':28} {I1_TRAIN_MAPE:>10.2f}% {I1_VAL_MAPE:>9.2f}% "
      f"{I1_TRAIN_R2:>10.3f} {I1_VAL_R2:>8.3f}")
print(f"{'Mixed Iter 1':28} {I1_MIX_TRAIN_MAPE:>10.2f}% {I1_MIX_VAL_MAPE:>9.2f}% "
      f"{I1_MIX_TRAIN_R2:>10.3f} {I1_MIX_VAL_R2:>8.3f}")
print(f"{'Hierarchical Iter 2':28} {I2_TRAIN_MAPE:>10.2f}% {I2_VAL_MAPE:>9.2f}% "
      f"{I2_TRAIN_R2:>10.3f} {I2_VAL_R2:>8.3f}")
print(f"{'Mixed Hierarchical Iter 2':28} {mape_train:>10.2f}% {mape_val:>9.2f}% "
      f"{r2_train:>10.3f} {r2_val:>8.3f}  ← FINAL")
print("─" * 72)

delta_mape = mape_val - I2_VAL_MAPE
delta_r2   = r2_val   - I2_VAL_R2
print(f"\nVs Hierarchical Iter 2:")
print(f"  Val MAPE delta : {delta_mape:+.2f} pts  "
      f"{'↓ BETTER' if delta_mape < 0 else '↑ WORSE'}")
print(f"  Val R²   delta : {delta_r2:+.3f}      "
      f"{'↑ BETTER' if delta_r2 > 0 else '↓ WORSE'}")

print("\n--- Residual summary ---")
print(f"{'':20} {'Train':>15} {'Val':>15}")
print(f"{'Mean':20} {resid_train.mean():>15,.0f} {resid_val.mean():>15,.0f}")
print(f"{'Std':20} {resid_train.std():>15,.0f} {resid_val.std():>15,.0f}")
print(f"{'Median |err|':20} {np.median(np.abs(resid_train)):>15,.0f} "
      f"{np.median(np.abs(resid_val)):>15,.0f}")

# ── Build df_trainval ─────────────────────────────────────────────────────────
df_trainval = pd.concat([train_df, val_df]).copy().reset_index(drop=True)
df_trainval["segment"]       = ["train"] * n_train + ["val"] * n_val
df_trainval["week_date"]     = df_trainval["week_date"]
df_trainval["pred"]          = pred_trainval
df_trainval["resid"]         = df_trainval[target_col].values - pred_trainval
df_trainval["controls_pred"] = scaler_y.inverse_transform(
                                   mu_controls.reshape(-1, 1)).ravel()

# LOB-level media contributions
for combo in ACTIVE_COMBOS:
    df_trainval[f"media_{combo}"] = media_contrib[combo]

# Funnel-level media aggregates
for funnel, combos in FUNNEL_COMBOS.items():
    df_trainval[f"media_{funnel}"] = sum(
        df_trainval[f"media_{combo}"] for combo in combos
    )

# Total media
df_trainval["total_media"] = sum(
    media_contrib[c] for c in ACTIVE_COMBOS
)

print(f"\n✓ df_trainval built — {len(df_trainval)} rows, "
      f"{len([c for c in df_trainval.columns if 'media_' in c])} media columns")

In [0]:
# ── CELL: Annual prediction error metrics — Final Model ───────────────────────

# ── Rebuild df_trainval if not in memory ──────────────────────────────────────
post = trace_iter2_mixed_hier.posterior

intercept_mean = float(post["intercept"].values.ravel().mean())
beta_mean      = post["beta"].values.reshape(-1, len(baseline_features)).mean(axis=0)

X_trainval  = np.vstack([X_train, X_val])
mu_controls = intercept_mean + X_trainval @ beta_mean
mu_media    = np.zeros(n_train + n_val)
media_contrib = {}

for funnel, combos in FUNNEL_COMBOS.items():
    for combo in combos:
        alpha_mean = float(post[f"alpha_{combo}"].values.ravel().mean())
        gamma_mean = float(post[f"gamma_{combo}"].values.ravel().mean())
        x_spend    = spend_scaled_iter2[combo]["trainval"]

        adstocked    = np.zeros(len(x_spend))
        adstocked[0] = x_spend[0]
        for t in range(1, len(x_spend)):
            adstocked[t] = x_spend[t] + alpha_mean * adstocked[t - 1]

        if funnel == "Consideration":
            k_mean   = float(post[f"k_{combo}"].values.ravel().mean())
            as_mean  = float(post[f"alpha_s_{combo}"].values.ravel().mean())
            x_safe   = np.maximum(adstocked, 1e-6)
            sat      = (x_safe ** as_mean) / (x_safe ** as_mean + k_mean ** as_mean)
        else:
            lam_mean = float(post[f"lam_{combo}"].values.ravel().mean())
            sat      = 1 - np.exp(-lam_mean * adstocked)

        contrib_scaled        = gamma_mean * sat
        mu_media             += contrib_scaled
        media_contrib[combo]  = contrib_scaled * scaler_y.scale_[0]

df_trainval = pd.concat([train_df, val_df]).copy().reset_index(drop=True)
df_trainval["segment"]       = ["train"] * n_train + ["val"] * n_val
df_trainval["pred"]          = scaler_y.inverse_transform(
                                   (mu_controls + mu_media).reshape(-1, 1)).ravel()
df_trainval["resid"]         = df_trainval[target_col].values - df_trainval["pred"].values
df_trainval["controls_pred"] = scaler_y.inverse_transform(
                                   mu_controls.reshape(-1, 1)).ravel()

for combo in ACTIVE_COMBOS:
    df_trainval[f"media_{combo}"] = media_contrib[combo]

for funnel, combos in FUNNEL_COMBOS.items():
    df_trainval[f"media_{funnel}"] = sum(
        df_trainval[f"media_{combo}"] for combo in combos
    )

actual_train_v = train_df[target_col].values
actual_val_v   = val_df[target_col].values
pred_train_v   = df_trainval["pred"].values[:n_train]
pred_val_v     = df_trainval["pred"].values[n_train:]

print("✓ df_trainval rebuilt")

# ── rest of the annual metrics cell continues below ───────────────────────────
df_trainval["year"] = df_trainval["week_date"].dt.year

annual_metrics = df_trainval.groupby("year").agg(
    actual    = (target_col, "sum"),
    predicted = ("pred",     "sum"),
    n_weeks   = ("pred",     "count"),
).reset_index()

annual_metrics["error"]         = annual_metrics["actual"] - annual_metrics["predicted"]
annual_metrics["abs_error"]     = annual_metrics["error"].abs()
annual_metrics["pct_error"]     = annual_metrics["error"] / annual_metrics["actual"] * 100
annual_metrics["abs_pct_error"] = annual_metrics["abs_error"] / annual_metrics["actual"] * 100

weekly_stats = df_trainval.groupby("year").agg(
    resid_mean       = ("resid", "mean"),
    resid_std        = ("resid", "std"),
    resid_median_abs = ("resid", lambda x: np.median(np.abs(x))),
).reset_index()

summary = annual_metrics.merge(weekly_stats, on="year")

val_years = df_trainval.loc[
    df_trainval["segment"] == "val", "week_date"
].dt.year.unique()

summary["segment"] = summary["year"].apply(
    lambda y: "Val" if y in val_years else "Train"
)

AUTOCORR_ADJ = 1.3
summary["annual_random_error"] = (
    summary["resid_std"] * np.sqrt(summary["n_weeks"]) * AUTOCORR_ADJ
)
summary["annual_random_error_pct"] = (
    summary["annual_random_error"] / summary["actual"] * 100
)
summary["annual_rmse"] = np.sqrt(
    summary["error"] ** 2 + summary["annual_random_error"] ** 2
)
summary["annual_rmse_pct"] = summary["annual_rmse"] / summary["actual"] * 100

print("=" * 95)
print("ANNUAL PREDICTION ERROR METRICS — Final Model (Iter 2 Mixed Hierarchical)")
print("=" * 95)
print(f"\n{'Year':<6} {'Seg':<6} {'Actual':>12} {'Predicted':>12} {'Error':>12} "
      f"{'Bias %':>8} {'|Err %|':>8} {'Wk Std':>12}")
print("─" * 95)
for _, row in summary.iterrows():
    print(f"{int(row['year']):<6} {row['segment']:<6} "
          f"{row['actual']:>12,.0f} {row['predicted']:>12,.0f} "
          f"{row['error']:>12,.0f} {row['pct_error']:>7.2f}% "
          f"{row['abs_pct_error']:>7.2f}% {row['resid_std']:>12,.0f}")

print("\n" + "=" * 95)
print("ANNUAL ERROR DECOMPOSITION (CLT-based)")
print("=" * 95)
print(f"\n{'Year':<6} {'Seg':<6} {'Bias':>12} {'Bias %':>8} "
      f"{'Random Err':>12} {'Rand %':>8} {'RMSE':>12} {'RMSE %':>8}")
print("─" * 95)
for _, row in summary.iterrows():
    print(f"{int(row['year']):<6} {row['segment']:<6} "
          f"{row['error']:>12,.0f} {row['pct_error']:>7.2f}% "
          f"{row['annual_random_error']:>12,.0f} "
          f"{row['annual_random_error_pct']:>7.2f}% "
          f"{row['annual_rmse']:>12,.0f} {row['annual_rmse_pct']:>7.2f}%")

print("\n" + "=" * 95)
print("SEGMENT SUMMARY")
print("=" * 95)
for seg in ["Train", "Val"]:
    seg_df = summary[summary["segment"] == seg]
    if len(seg_df) == 0:
        continue
    total_actual    = seg_df["actual"].sum()
    total_predicted = seg_df["predicted"].sum()
    total_error     = seg_df["error"].sum()
    total_bias_pct  = total_error / total_actual * 100
    avg_rmse_pct    = seg_df["annual_rmse_pct"].mean()

    print(f"\n{seg}:")
    print(f"  Total actual         : {total_actual:>15,.0f}")
    print(f"  Total predicted      : {total_predicted:>15,.0f}")
    print(f"  Total bias           : {total_error:>15,.0f}  ({total_bias_pct:+.2f}%)")
    print(f"  Avg annual RMSE      : {avg_rmse_pct:>14.2f}%")

val_rmse_pct = summary[summary["segment"] == "Val"]["annual_rmse_pct"].mean()
val_actual   = summary[summary["segment"] == "Val"]["actual"].mean()
mdu          = 2 * val_rmse_pct
print(f"\nMinimum detectable uplift (2× avg annual RMSE, val period):")
print(f"  {mdu:.1f}% of annual attendance ≈ {val_actual * mdu / 100:,.0f} attendees/year")

In [0]:
# Ensure post points to final model
post = trace_iter2_mixed_hier.posterior

intercept_mean = float(post["intercept"].values.ravel().mean())
beta_mean      = post["beta"].values.reshape(-1, len(baseline_features)).mean(axis=0)

X_trainval  = np.vstack([X_train, X_val])
mu_controls = intercept_mean + X_trainval @ beta_mean

mu_media      = np.zeros(n_train + n_val)
media_contrib = {}

for funnel, combos in FUNNEL_COMBOS.items():
    for combo in combos:
        alpha_mean = float(post[f"alpha_{combo}"].values.ravel().mean())
        gamma_mean = float(post[f"gamma_{combo}"].values.ravel().mean())

        x_spend = spend_scaled_iter2[combo]["trainval"]

        adstocked_np    = np.zeros(len(x_spend))
        adstocked_np[0] = x_spend[0]
        for t in range(1, len(x_spend)):
            adstocked_np[t] = x_spend[t] + alpha_mean * adstocked_np[t - 1]

        if funnel == "Consideration":
            k_mean       = float(post[f"k_{combo}"].values.ravel().mean())
            alpha_s_mean = float(post[f"alpha_s_{combo}"].values.ravel().mean())
            x_safe       = np.maximum(adstocked_np, 1e-6)
            saturated_np = (x_safe ** alpha_s_mean) / (
                x_safe ** alpha_s_mean + k_mean ** alpha_s_mean
            )
        else:
            lam_mean     = float(post[f"lam_{combo}"].values.ravel().mean())
            saturated_np = 1 - np.exp(-lam_mean * adstocked_np)

        contrib_scaled        = gamma_mean * saturated_np
        mu_media             += contrib_scaled
        media_contrib[combo]  = contrib_scaled * scaler_y.scale_[0]

# Rebuild df_trainval
df_trainval = pd.concat([train_df, val_df]).copy().reset_index(drop=True)
df_trainval["segment"]       = ["train"] * n_train + ["val"] * n_val
df_trainval["pred"]          = scaler_y.inverse_transform(
                                   (mu_controls + mu_media).reshape(-1, 1)).ravel()
df_trainval["resid"]         = df_trainval[target_col].values - df_trainval["pred"].values
df_trainval["controls_pred"] = scaler_y.inverse_transform(
                                   mu_controls.reshape(-1, 1)).ravel()

# LOB-level media columns
for combo in ACTIVE_COMBOS:
    df_trainval[f"media_{combo}"] = media_contrib[combo]

# Funnel-level aggregates
for funnel, combos in FUNNEL_COMBOS.items():
    df_trainval[f"media_{funnel}"] = sum(
        df_trainval[f"media_{combo}"] for combo in combos
    )

df_trainval["total_media"] = sum(media_contrib[c] for c in ACTIVE_COMBOS)

# Recompute prediction arrays
pred_trainval  = df_trainval["pred"].values
pred_train_v   = pred_trainval[:n_train]
pred_val_v     = pred_trainval[n_train:]
actual_train_v = train_df[target_col].values
actual_val_v   = val_df[target_col].values
resid_train    = actual_train_v - pred_train_v
resid_val      = actual_val_v   - pred_val_v

baseline_orig   = intercept_mean * scaler_y.scale_[0] + scaler_y.mean_[0]
controls_effect = (mu_controls - intercept_mean) * scaler_y.scale_[0]

print("df_trainval rebuilt from final model trace ✓")
print(f"Media columns: {[c for c in df_trainval.columns if c.startswith('media_')]}")

# ── ORIG_LOB_CONTRIB — must be defined before Cells 28/29 ────────────────────
ORIG_LOB_CONTRIB = {}
for funnel, combos in FUNNEL_COMBOS.items():
    for combo in combos:
        lob_group = combo.split("__")[1]
        for orig_lob in LOB_GROUP_MAP[lob_group]:
            # Spend share proportion for this original LOB within its group
            group_cols = [
                c for c in media_meta.loc[
                    (media_meta["funnel"] == funnel) &
                    (media_meta["lob"].isin(LOB_GROUP_MAP[lob_group])),
                    "col_name"
                ].tolist()
            ]
            orig_cols = media_meta.loc[
                (media_meta["funnel"] == funnel) &
                (media_meta["lob"] == orig_lob),
                "col_name"
            ].tolist()

            if not orig_cols:
                continue

            group_total = df_trainval[group_cols].sum(axis=1).sum()
            orig_total  = df_trainval[orig_cols].sum(axis=1).sum()
            share       = orig_total / group_total if group_total > 0 else 0.0

            if share == 0:
                continue

            contrib_col = f"contrib_{funnel}__{orig_lob}"
            df_trainval[contrib_col]  = df_trainval[f"media_{combo}"] * share
            ORIG_LOB_CONTRIB[contrib_col] = {
                "funnel"  : funnel,
                "combo"   : combo,
                "orig_lob": orig_lob,
                "share"   : share,
            }

print(f"ORIG_LOB_CONTRIB built: {len(ORIG_LOB_CONTRIB)} entries")
for col, meta in ORIG_LOB_CONTRIB.items():
    print(f"  {col:<45} share={meta['share']:.3f}")

    # ── CELL: Saturation and adstock utility functions ────────────────────────────

# ── CELL: Saturation and adstock utility functions ────────────────────────────

def logistic_sat(x, lam):
    return 1 - np.exp(-lam * x)

def hill_sat(x, k, alpha_s):
    x_safe = np.maximum(x, 1e-6)
    return (x_safe ** alpha_s) / (x_safe ** alpha_s + k ** alpha_s)

def steady_state_adstock(s_norm, alpha):
    return s_norm / (1 - alpha)

def saturation_point_logistic(lam, threshold=0.95):
    return -np.log(1 - threshold) / lam

def saturation_point_hill(k, alpha_s, threshold=0.95):
    return k * (threshold / (1 - threshold)) ** (1 / alpha_s)

def marginal_logistic(x_norm, lam, gamma, scale, max_spend):
    return (lam * np.exp(-lam * x_norm) * gamma * scale) / max_spend

def marginal_hill(x_norm, k, alpha_s, gamma, scale, max_spend):
    x_safe = np.maximum(x_norm, 1e-6)
    deriv  = (alpha_s * k ** alpha_s * x_safe ** (alpha_s - 1)) / \
             (x_safe ** alpha_s + k ** alpha_s) ** 2
    return (deriv * gamma * scale) / max_spend

# ── Shared grids ──────────────────────────────────────────────────────────────
x_grid       = np.linspace(0, 1.5, 300)   # normalised spend grid [0, 1.5x max]

YEAR_LINESTYLES = {
    2023: {"color": "#2ecc71", "linestyle": "-",  "linewidth": 1.2},
    2024: {"color": "#e67e22", "linestyle": "--", "linewidth": 1.2},
    2025: {"color": "#9b59b6", "linestyle": ":",  "linewidth": 1.5},
}

YEAR_MARKERS = {
    2023: {"color": "#2ecc71", "marker": "o"},
    2024: {"color": "#e67e22", "marker": "s"},
    2025: {"color": "#9b59b6", "marker": "D"},
}

print("✓ Saturation functions, x_grid, YEAR_LINESTYLES, YEAR_MARKERS defined")

# VISUALIZATIONS

In [0]:
from scipy.stats import beta as beta_dist, halfnorm
import numpy as np

post = trace_iter2_mixed_hier.posterior

ALPHA_MU_PRIOR = {
    "Awareness"    : {"alpha": 7, "beta": 2},
    "Consideration": {"alpha": 3, "beta": 3},
    "Conversion"   : {"alpha": 2, "beta": 5},
}

MU_GAMMA_PRIOR = {
    "Awareness"    : 0.5,
    "Consideration": 0.3,
    "Conversion"   : 0.6,
}

x_alpha  = np.linspace(0, 1,   300)
x_lam    = np.linspace(0, 5,   300)
x_gamma  = np.linspace(0, 2,   300)
x_k      = np.linspace(0, 2,   300)
x_alphas = np.linspace(0, 6,   300)
x_kappa  = np.linspace(0, 10,  300)

FUNNEL_COLORS = {
    "Awareness"    : CPX_NAVY,
    "Consideration": CPX_RED,
    "Conversion"   : CPX_GOLD,
}

# ── Chart 1: Hyperparameters ──────────────────────────────────────────────────
fig, axes = plt.subplots(3, 4, figsize=(22, 14))

for row, funnel in enumerate(["Awareness", "Consideration", "Conversion"]):
    color = FUNNEL_COLORS[funnel]
    ap    = ALPHA_MU_PRIOR[funnel]

    # mu_alpha
    ax = axes[row, 0]
    ax.plot(x_alpha, beta_dist.pdf(x_alpha, ap["alpha"], ap["beta"]),
            color=CPX_DGREY, linewidth=1.5, linestyle="--", label="Prior")
    ax.hist(post[f"mu_alpha_{funnel}"].values.ravel(), bins=60, density=True,
            color=color, alpha=0.6, label="Posterior")
    ax.set_title(f"{funnel} — mu_alpha (pooled decay mean)",
                 fontweight="bold", color=CPX_NAVY)
    ax.set_xlabel("mu_alpha")
    ax.legend(fontsize=7)
    ax.grid(False)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    # kappa_alpha
    ax = axes[row, 1]
    ax.plot(x_kappa, halfnorm.pdf(x_kappa, scale=2),
            color=CPX_DGREY, linewidth=1.5, linestyle="--", label="Prior")
    ax.hist(post[f"kappa_alpha_{funnel}"].values.ravel(), bins=60, density=True,
            color=color, alpha=0.6, label="Posterior")
    ax.set_title(f"{funnel} — kappa_alpha (pooling concentration)",
                 fontweight="bold", color=CPX_NAVY)
    ax.set_xlabel("kappa_alpha")
    ax.legend(fontsize=7)
    ax.grid(False)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    # mu_gamma
    ax = axes[row, 2]
    ax.plot(x_gamma, halfnorm.pdf(x_gamma, scale=MU_GAMMA_PRIOR[funnel]),
            color=CPX_DGREY, linewidth=1.5, linestyle="--", label="Prior")
    ax.hist(post[f"mu_gamma_{funnel}"].values.ravel(), bins=60, density=True,
            color=color, alpha=0.6, label="Posterior")
    ax.set_title(f"{funnel} — mu_gamma (pooled effect mean)",
                 fontweight="bold", color=CPX_NAVY)
    ax.set_xlabel("mu_gamma")
    ax.legend(fontsize=7)
    ax.grid(False)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    # Consideration Hill hyperpriors / blank for others
    ax = axes[row, 3]
    if funnel == "Consideration":
        ax.plot(x_k, halfnorm.pdf(x_k, scale=0.5),
                color=CPX_DGREY, linewidth=1.5, linestyle="--", label="Prior (mu_k)")
        ax.hist(post["mu_k_Consideration"].values.ravel(), bins=60, density=True,
                color=color, alpha=0.6, label="Posterior (mu_k)")
        ax2 = ax.twinx()
        ax2.plot(x_alphas, halfnorm.pdf(x_alphas, scale=1.5),
                 color=CPX_DGREY, linewidth=1.5, linestyle=":",
                 label="Prior (mu_alpha_s)")
        ax2.hist(post["mu_alpha_s_Consideration"].values.ravel(), bins=60,
                 density=True, color=color, alpha=0.3,
                 label="Posterior (mu_alpha_s)")
        ax2.axvline(1.0, color=CPX_DGREY, linewidth=1, linestyle=":",
                    label="α_s=1 (hyperbolic)")
        ax2.set_ylabel("mu_alpha_s density", fontsize=7)
        ax2.legend(fontsize=6, loc="upper right")
        ax.set_title("Consideration — mu_k & mu_alpha_s (Hill hyperpriors)",
                     fontweight="bold", color=CPX_NAVY)
        ax.set_xlabel("value")
        ax.legend(fontsize=6, loc="upper left")
    else:
        ax.text(0.5, 0.5, "No Hill\nhyperpriors",
                ha="center", va="center", transform=ax.transAxes,
                fontsize=10, color=CPX_DGREY, fontstyle="italic")
        ax.set_title(f"{funnel} — Hill hyperpriors",
                     fontweight="bold", color=CPX_NAVY)
    ax.grid(False)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

fig.suptitle(
    "Prior vs Posterior — Hyperparameters (Final Model: Iter 2 Mixed Hierarchical)\n"
    "Rows = Funnels  |  Cols = mu_alpha, kappa_alpha, mu_gamma, Hill hyperpriors",
    fontsize=13, fontweight="bold", color=CPX_NAVY
)
plt.tight_layout()
save_fig(fig, "I2_D1_Hyperparameter_Posteriors.png")
plt.show()

# # ── Chart 2: LOB-level alpha and gamma per funnel ─────────────────────────────
# for funnel, combos in FUNNEL_COMBOS.items():
#     color = FUNNEL_COLORS[funnel]
#     ap    = ALPHA_MU_PRIOR[funnel]
#     n_col = len(combos)

#     fig, axes = plt.subplots(3, n_col, figsize=(6 * n_col, 14))
#     if n_col == 1:
#         axes = axes.reshape(-1, 1)

#     for col, combo in enumerate(combos):
#         lob_label = combo.split("__")[1].replace("_", " ")

#         # Row 0: alpha
#         ax = axes[0, col]
#         ax.plot(x_alpha, beta_dist.pdf(x_alpha, ap["alpha"], ap["beta"]),
#                 color=CPX_DGREY, linewidth=1.5, linestyle="--", label="Hyperprior")
#         ax.hist(post[f"alpha_{combo}"].values.ravel(), bins=60, density=True,
#                 color=color, alpha=0.6, label="Posterior")
#         ax.set_title(f"{lob_label}\nalpha (decay)",
#                      fontweight="bold", color=CPX_NAVY)
#         ax.set_xlabel("alpha")
#         ax.legend(fontsize=7)
#         ax.grid(False)
#         ax.spines["top"].set_visible(False)
#         ax.spines["right"].set_visible(False)

#         # Row 1: saturation param
#         ax = axes[1, col]
#         if funnel == "Consideration":
#             ax.plot(x_k, halfnorm.pdf(x_k, scale=0.5),
#                     color=CPX_DGREY, linewidth=1.5, linestyle="--", label="Prior")
#             ax.hist(post[f"k_{combo}"].values.ravel(), bins=60, density=True,
#                     color=color, alpha=0.6, label="Posterior")
#             ax.set_title(f"{lob_label}\nk (inflection)",
#                          fontweight="bold", color=CPX_NAVY)
#             ax.set_xlabel("k")
#         else:
#             ax.plot(x_lam, halfnorm.pdf(x_lam, scale=1),
#                     color=CPX_DGREY, linewidth=1.5, linestyle="--", label="Prior")
#             ax.hist(post[f"lam_{combo}"].values.ravel(), bins=60, density=True,
#                     color=color, alpha=0.6, label="Posterior")
#             ax.set_title(f"{lob_label}\nlam (saturation)",
#                          fontweight="bold", color=CPX_NAVY)
#             ax.set_xlabel("lam")
#         ax.legend(fontsize=7)
#         ax.grid(False)
#         ax.spines["top"].set_visible(False)
#         ax.spines["right"].set_visible(False)

#         # Row 2: gamma
#         ax = axes[2, col]
#         ax.plot(x_gamma, halfnorm.pdf(x_gamma, scale=MU_GAMMA_PRIOR[funnel]),
#                 color=CPX_DGREY, linewidth=1.5, linestyle="--", label="Prior")
#         ax.hist(post[f"gamma_{combo}"].values.ravel(), bins=60, density=True,
#                 color=color, alpha=0.6, label="Posterior")
#         ax.set_title(f"{lob_label}\ngamma (effect size)",
#                      fontweight="bold", color=CPX_NAVY)
#         ax.set_xlabel("gamma")
#         ax.legend(fontsize=7)
#         ax.grid(False)
#         ax.spines["top"].set_visible(False)
#         ax.spines["right"].set_visible(False)

#     fig.suptitle(
#         f"Prior vs Posterior — {funnel} LOB Parameters (Final Model)\n"
#         "Rows = alpha, saturation, gamma  |  Cols = LOBs",
#         fontsize=13, fontweight="bold", color=CPX_NAVY
#     )
#     plt.tight_layout()
#     save_fig(fig, f"I2_D2_{funnel}_LOB_Posteriors.png")
#     plt.show()

In [0]:
# ── CELL: Adstock decay curves — Final Model ──────────────────────────────────

FUNNEL_COLORS = {
    "Awareness"    : CPX_NAVY,
    "Consideration": CPX_RED,
    "Conversion"   : CPX_GOLD,
}

WEEKS_OUT = 16
weeks     = np.arange(WEEKS_OUT)

post = trace_iter2_mixed_hier.posterior

# ── Chart 1: Funnel level (mu_alpha hyperpriors) ──────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))

for funnel in ["Awareness", "Consideration", "Conversion"]:
    samples    = post[f"mu_alpha_{funnel}"].values.ravel()
    alpha_mean = float(samples.mean())
    alpha_lo   = float(np.quantile(samples, 0.025))
    alpha_hi   = float(np.quantile(samples, 0.975))

    decay_mean = np.array([alpha_mean ** t for t in range(WEEKS_OUT)])
    decay_lo   = np.array([alpha_lo   ** t for t in range(WEEKS_OUT)])
    decay_hi   = np.array([alpha_hi   ** t for t in range(WEEKS_OUT)])

    ax.plot(weeks, decay_mean, color=FUNNEL_COLORS[funnel], linewidth=2,
            label=f"{funnel} (μ_α={alpha_mean:.3f})")
    ax.fill_between(weeks, decay_lo, decay_hi,
                    color=FUNNEL_COLORS[funnel], alpha=0.15)

ax.axhline(0.05, color=CPX_DGREY, linewidth=0.8, linestyle="--", alpha=0.6,
           label="5% threshold")
ax.set_xlabel("Weeks after spend")
ax.set_ylabel("Remaining adstock effect (relative)")
ax.set_title(
    "Adstock Decay Curves by Funnel (Final Model — Funnel Level)\n"
    "Shaded = 95% credible interval  |  Dashed = 5% threshold",
    fontsize=11, fontweight="bold", color=CPX_NAVY,
)
ax.legend(fontsize=9)
ax.grid(False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
save_fig(fig, "I2_D7_Adstock_Decay_Funnel.png")
plt.show()

# # ── Chart 2: LOB level (individual alpha posteriors) ──────────────────────────
# LOB_LABELS = {
#     "Awareness__Premium_Brand"    : "Awareness — Premium & Brand",
#     "Consideration__Premium_Brand": "Consideration — Premium & Brand",
#     "Consideration__GE"           : "Consideration — G&E",
#     "Consideration__Prepaid"      : "Consideration — Prepaid",
#     "Conversion__Film_Marketing"  : "Conversion — Film Marketing",
#     "Conversion__Premium_Brand"   : "Conversion — Premium & Brand",
#     "Conversion__Cineclub"        : "Conversion — Cineclub",
# }

# LOB_LINESTYLES = {
#     "Awareness__Premium_Brand"    : "-",
#     "Consideration__Premium_Brand": "-",
#     "Consideration__GE"           : "--",
#     "Consideration__Prepaid"      : ":",
#     "Conversion__Film_Marketing"  : "-",
#     "Conversion__Premium_Brand"   : "--",
#     "Conversion__Cineclub"        : ":",
# }

# fig, ax = plt.subplots(figsize=(12, 6))

# for combo in ACTIVE_COMBOS:
#     funnel     = combo.split("__")[0]
#     samples    = post[f"alpha_{combo}"].values.ravel()
#     alpha_mean = float(samples.mean())
#     alpha_lo   = float(np.quantile(samples, 0.025))
#     alpha_hi   = float(np.quantile(samples, 0.975))

#     decay_mean = np.array([alpha_mean ** t for t in range(WEEKS_OUT)])
#     decay_lo   = np.array([alpha_lo   ** t for t in range(WEEKS_OUT)])
#     decay_hi   = np.array([alpha_hi   ** t for t in range(WEEKS_OUT)])

#     ax.plot(weeks, decay_mean,
#             color=FUNNEL_COLORS[funnel],
#             linewidth=2,
#             linestyle=LOB_LINESTYLES[combo],
#             label=f"{LOB_LABELS[combo]} (α={alpha_mean:.3f})")
#     ax.fill_between(weeks, decay_lo, decay_hi,
#                     color=FUNNEL_COLORS[funnel], alpha=0.08)

# ax.axhline(0.05, color=CPX_DGREY, linewidth=0.8, linestyle="--", alpha=0.6,
#            label="5% threshold")
# ax.set_xlabel("Weeks after spend")
# ax.set_ylabel("Remaining adstock effect (relative)")
# ax.set_title(
#     "Adstock Decay Curves by LOB (Final Model — LOB Level)\n"
#     "Colour = Funnel  |  Line style = LOB  |  Shaded = 95% credible interval",
#     fontsize=11, fontweight="bold", color=CPX_NAVY,
# )
# ax.legend(fontsize=8, ncol=2, loc="upper right",
#           framealpha=0.9, edgecolor=CPX_LGREY)
# ax.grid(False)
# ax.spines["top"].set_visible(False)
# ax.spines["right"].set_visible(False)
# plt.tight_layout()
# save_fig(fig, "I2_D8_Adstock_Decay_LOB.png")
# plt.show()

In [0]:
# ── CELL: Validation diagnostics — Final Model (Iter 2 Mixed Hierarchical) ────

mape_train = mean_absolute_percentage_error(actual_train_v, pred_train_v) * 100
mape_val   = mean_absolute_percentage_error(actual_val_v,   pred_val_v)   * 100
r2_train   = r2_score(actual_train_v, pred_train_v)
r2_val     = r2_score(actual_val_v,   pred_val_v)

val_dates     = val_df["week_date"].values
y_val_actual  = actual_val_v
y_val_pred    = pred_val_v
residuals_val = y_val_actual - y_val_pred

# ── Chart 1: Actual vs Predicted (full train + val) ───────────────────────────
fig, ax = plt.subplots(figsize=(14, 5))

ax.plot(df_trainval["week_date"], df_trainval[target_col] / 1e6,
        color="black", linewidth=1.5, label="Actual")
ax.plot(df_trainval["week_date"], df_trainval["pred"] / 1e6,
        color=CPX_NAVY, linewidth=1.5, linestyle="--", label="Predicted")
ax.plot(df_trainval["week_date"], df_trainval["controls_pred"] / 1e6,
        color=CPX_DGREY, linewidth=1.0, linestyle=":", alpha=0.7,
        label="Baseline (controls only)")
ax.axvspan(val_df["week_date"].iloc[0], val_df["week_date"].iloc[-1],
           alpha=0.08, color=CPX_RED, label="Validation period")
ax.set_ylabel("Weekly Attendance (M)")
ax.set_title(
    f"Actual vs Predicted (Final Model: Iter 2 Mixed Hierarchical)  —  "
    f"Train MAPE {mape_train:.1f}%  R² {r2_train:.3f}  |  "
    f"Val MAPE {mape_val:.1f}%  R² {r2_val:.3f}",
    fontsize=11, fontweight="bold", color=CPX_NAVY,
)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.1f}M"))
ax.legend(fontsize=8, ncol=2)
ax.grid(False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
save_fig(fig, "I2_D3_Actual_vs_Predicted.png")
plt.show()

# ── Chart 2: Validation window — actual vs predicted with fill ────────────────
fig, ax = plt.subplots(figsize=(14, 5))

ax.plot(val_dates, y_val_actual / 1e6, label="Actual",
        color="steelblue", linewidth=2)
ax.plot(val_dates, y_val_pred / 1e6,   label="Predicted",
        color="darkorange", linewidth=2, linestyle="--")
ax.fill_between(val_dates, y_val_actual / 1e6, y_val_pred / 1e6,
                where=(y_val_actual >= y_val_pred),
                alpha=0.2, color="green", label="Under-predicted")
ax.fill_between(val_dates, y_val_actual / 1e6, y_val_pred / 1e6,
                where=(y_val_actual <  y_val_pred),
                alpha=0.2, color="red", label="Over-predicted")
ax.set_ylabel("Weekly Attendance (M)")
ax.set_title(
    "Validation Window — Actual vs Predicted (Final Model: Iter 2 Mixed Hierarchical)",
    fontsize=11, fontweight="bold", color=CPX_NAVY,
)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.2f}M"))
ax.legend(fontsize=9)
ax.grid(False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
save_fig(fig, "I2_D4_Val_Actual_vs_Predicted.png")
plt.show()

# ── Chart 3: Residuals over time ──────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 5))

ax.plot(
    df_trainval.loc[df_trainval["segment"] == "train", "week_date"],
    resid_train / 1e3,
    color=CPX_NAVY, linewidth=1, label="Train residual",
)
ax.bar(
    df_trainval.loc[df_trainval["segment"] == "val", "week_date"],
    resid_val / 1e3,
    color=CPX_RED, alpha=0.7, width=5, label="Val residual",
)
ax.axhline(0, color="black", linewidth=0.8)
ax.axhline( resid_train.std() / 1e3, color=CPX_DGREY, linewidth=0.8,
            linestyle="--", alpha=0.6,
            label=f"±1σ train ({resid_train.std()/1e3:,.0f}K)")
ax.axhline(-resid_train.std() / 1e3, color=CPX_DGREY, linewidth=0.8,
            linestyle="--", alpha=0.6)
ax.axhline(residuals_val.mean() / 1e3, color=CPX_RED, linewidth=1.2,
            linestyle="--",
            label=f"Val mean ({residuals_val.mean()/1e3:,.0f}K)")
ax.set_ylabel("Residual (000s)")
ax.set_title(
    "Residuals Over Time — Target: Random, No Structure (Final Model)",
    fontsize=11, fontweight="bold", color=CPX_NAVY,
)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:,.0f}K"))
ax.legend(fontsize=8, ncol=2)
ax.grid(False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
save_fig(fig, "I2_D5_Residuals_Over_Time.png")
plt.show()

# ── Chart 4: Predicted vs Actual scatter ──────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 7))

ax.scatter(actual_train_v / 1e6, pred_train_v / 1e6,
           alpha=0.5, s=25, color=CPX_NAVY, label="Train")
ax.scatter(actual_val_v / 1e6, pred_val_v / 1e6,
           alpha=0.9, s=60, color=CPX_RED, marker="D", label="Val")

lim_lo = min(actual_train_v.min(), pred_train_v.min()) / 1e6
lim_hi = max(actual_train_v.max(), pred_train_v.max()) / 1e6
ax.plot([lim_lo, lim_hi], [lim_lo, lim_hi],
        "k--", linewidth=1, label="Perfect fit")

ax.set_xlabel("Actual Attendance (M)")
ax.set_ylabel("Predicted Attendance (M)")
ax.set_title(
    "Predicted vs Actual Scatter (Final Model)",
    fontsize=11, fontweight="bold", color=CPX_NAVY,
)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.1f}M"))
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.1f}M"))
ax.legend(fontsize=8)
ax.grid(False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
save_fig(fig, "I2_D6_Scatter_Predicted_vs_Actual.png")
plt.show()

In [0]:
# ── CELL 15: Business — Contribution decomposition over time (Final Model) ────

FUNNEL_COLORS = {
    "Awareness"    : CPX_NAVY,
    "Consideration": CPX_RED,
    "Conversion"   : CPX_GOLD,
}

fig, ax = plt.subplots(figsize=(18, 7))

ax.fill_between(
    df_trainval["week_date"], 0, baseline_orig / 1e6,
    alpha=0.9, color=CPX_LGREY,
    label=f"Intercept baseline ({baseline_orig/1e6:.2f}M avg)",
)

ax.fill_between(
    df_trainval["week_date"],
    baseline_orig / 1e6,
    (baseline_orig + controls_effect) / 1e6,
    alpha=0.7, color=CPX_DGREY, label="Controls effect",
)

stack_top = (baseline_orig + controls_effect).copy()
for funnel, combos in FUNNEL_COMBOS.items():
    media_f = df_trainval[f"media_{funnel}"].values
    ax.fill_between(
        df_trainval["week_date"],
        stack_top / 1e6,
        (stack_top + media_f) / 1e6,
        alpha=0.85, color=FUNNEL_COLORS[funnel],
        label=f"{funnel} media",
    )
    stack_top += media_f

ax.plot(
    df_trainval["week_date"],
    df_trainval[target_col] / 1e6,
    color="black", linewidth=1.5, label="Actual", zorder=5,
)

ax.axvspan(
    val_df["week_date"].iloc[0], val_df["week_date"].iloc[-1],
    alpha=0.06, color=CPX_RED,
)
ax.annotate(
    "← Validation",
    xy=(val_df["week_date"].iloc[0], ax.get_ylim()[1] * 0.95),
    fontsize=8, color=CPX_RED, fontstyle="italic",
)

ax.set_ylabel("Weekly Attendance (M)", color=CPX_NAVY)
ax.set_title(
    "EXH MMM — Attendance Decomposition by Component (MH)\n"
    "Stacked: Intercept → Controls → Media by Funnel  |  Line = Actual",
    fontsize=12, fontweight="bold", color=CPX_NAVY,
)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.1f}M"))
ax.legend(fontsize=8, ncol=4, loc="upper left")
ax.grid(False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
save_fig(fig, "I2_B1_Contribution_Decomposition_Final.png")
plt.show()

In [0]:
# ── CELL: Annual decomposition — by Funnel then by Original LOB ───────────────

import matplotlib as mpl
import matplotlib.patheffects as pe
from matplotlib.patches import Patch

mpl.rcParams['hatch.linewidth'] = 2.0
mpl.rcParams['hatch.color']     = 'white'

LIGHT_COLORS = {CPX_LGREY, CPX_GOLD}


def label(ax, x_pos, bottom, height, pct, color, min_height=0.5):
    if abs(height) / 1e6 < min_height:
        return
    txt_color = "black" if color in LIGHT_COLORS else "white"
    ax.text(
        x_pos, (bottom + height / 2) / 1e6,
        f"{pct:.0f}%",
        ha="center", va="center",
        fontsize=8, fontweight="bold", color=txt_color,
        path_effects=[pe.withStroke(
            linewidth=2,
            foreground="white" if txt_color == "black" else "black"
        )],
    )


def format_annual_ax(ax, years_avail, actual_vals, title):
    ax.set_xticks(np.arange(len(years_avail)))
    ax.set_xticklabels(years_avail)
    ax.set_ylabel("Annual Attendance (M)")
    ax.set_title(title, fontsize=12, fontweight="bold", color=CPX_NAVY)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0f}M"))
    ax.grid(False)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)


# ── Step 1: Build annual table ────────────────────────────────────────────────
df_trainval["year"] = df_trainval["week_date"].dt.year

agg_dict = {target_col: "sum"}
for funnel, combos in FUNNEL_COMBOS.items():
    agg_dict[f"media_{funnel}"] = "sum"
for col in ORIG_LOB_CONTRIB:
    agg_dict[col] = "sum"

annual = df_trainval.groupby("year").agg(agg_dict).round(0)

years_avail  = sorted(annual.index.tolist())
actual_vals  = annual.loc[years_avail, target_col].values
x            = np.arange(len(years_avail))
w            = 0.6

awareness_vals     = annual.loc[years_avail, "media_Awareness"].values
consideration_vals = annual.loc[years_avail, "media_Consideration"].values
conversion_vals    = annual.loc[years_avail, "media_Conversion"].values

baseline_vals = (
    actual_vals
    - awareness_vals
    - consideration_vals
    - conversion_vals
)

# ══════════════════════════════════════════════════════════════════════════════
# Chart 1 — By Funnel
# ══════════════════════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(10, 6))

# Baseline
ax.bar(x, baseline_vals / 1e6, width=w, color=CPX_LGREY,
       alpha=0.9, edgecolor="white", linewidth=0.5, label="Baseline (non-media)")
for i, (v, b) in enumerate(zip(baseline_vals, np.zeros(len(years_avail)))):
    label(ax, x[i], b, v, v / actual_vals[i] * 100, CPX_LGREY)

# Awareness
ax.bar(x, awareness_vals / 1e6, width=w, bottom=baseline_vals / 1e6,
       color=CPX_NAVY, alpha=0.9, edgecolor="white", linewidth=0.5,
       label="Awareness")
for i, (v, b) in enumerate(zip(awareness_vals, baseline_vals)):
    label(ax, x[i], b, v, v / actual_vals[i] * 100, CPX_NAVY, min_height=1.5)

# Consideration
consideration_bottom = baseline_vals + awareness_vals
ax.bar(x, consideration_vals / 1e6, width=w,
       bottom=consideration_bottom / 1e6,
       color=CPX_RED, alpha=0.9, edgecolor="white", linewidth=0.5,
       label="Consideration")
for i, (v, b) in enumerate(zip(consideration_vals, consideration_bottom)):
    label(ax, x[i], b, v, v / actual_vals[i] * 100, CPX_RED, min_height=1.5)

# Conversion
conversion_bottom = consideration_bottom + consideration_vals
ax.bar(x, conversion_vals / 1e6, width=w,
       bottom=conversion_bottom / 1e6,
       color=CPX_GOLD, alpha=0.9, edgecolor="white", linewidth=0.5,
       label="Conversion")
for i, (v, b) in enumerate(zip(conversion_vals, conversion_bottom)):
    label(ax, x[i], b, v, v / actual_vals[i] * 100, CPX_GOLD, min_height=1.5)

# Actual dots
ax.scatter(x, actual_vals / 1e6, color="black", zorder=5,
           s=60, label="Actual", marker="D")

handles, labels_ = ax.get_legend_handles_labels()
ax.legend(handles, labels_, fontsize=8, ncol=2)
format_annual_ax(ax, years_avail, actual_vals,
    "Annual Attendance Decomposition by Funnel (Final Model)")
plt.tight_layout()
save_fig(fig, "I2_B6_Annual_Decomposition_Funnel.png")
plt.show()

In [0]:
# ── CELL: Annual media contribution — original LOBs, one bar each, grouped by year ──

ORIG_LOB_LIST = []
for funnel, combos in FUNNEL_COMBOS.items():
    for combo in combos:
        lob_group = combo.split("__")[1]
        for orig_lob in LOB_GROUP_MAP[lob_group]:
            contrib_col = f"contrib_{funnel}__{orig_lob}"
            if contrib_col in ORIG_LOB_CONTRIB:
                ORIG_LOB_LIST.append({
                    "funnel"     : funnel,
                    "orig_lob"   : orig_lob,
                    "contrib_col": contrib_col,
                    "color"      : FUNNEL_COLORS[funnel],
                    "label"      : orig_lob,
                })

# Deduplicate — same orig_lob can appear in multiple combos
seen       = set()
ORIG_LOBS  = []
for item in ORIG_LOB_LIST:
    key = f"{item['funnel']}__{item['orig_lob']}"
    if key not in seen:
        seen.add(key)
        ORIG_LOBS.append(item)

years_avail  = sorted(annual.index.tolist())
actual_vals  = annual.loc[years_avail, target_col].values
n_years      = len(years_avail)
n_lobs       = len(ORIG_LOBS)

bar_w        = 0.06
group_gap    = 1.4
year_centers = np.arange(n_years) * group_gap
total_w      = bar_w * n_lobs
offsets      = np.linspace(
    -total_w / 2 + bar_w / 2,
     total_w / 2 - bar_w / 2,
    n_lobs
)

fig, ax = plt.subplots(figsize=(16, 9))

legend_handles = {"Awareness": None, "Consideration": None, "Conversion": None}

for l_idx, lob in enumerate(ORIG_LOBS):
    vals  = annual.loc[years_avail, lob["contrib_col"]].values
    bar_x = year_centers + offsets[l_idx]

    bars = ax.bar(bar_x, vals / 1e6, width=bar_w * 0.9,
                  color=lob["color"], alpha=0.85,
                  edgecolor="white", linewidth=0.4)

    # Store one handle per funnel for legend
    if legend_handles[lob["funnel"]] is None:
        legend_handles[lob["funnel"]] = bars

    # Value label on top
    for bx, v in zip(bar_x, vals):
        if v / 1e6 > 0.05:
            ax.text(bx, v / 1e6 + 0.01,
                    f"{v/1e6:.1f}M",
                    ha="center", va="bottom",
                    fontsize=7, color=CPX_DGREY,
                    fontweight="bold", rotation=90)

# LOB name labels below bars
for l_idx, lob in enumerate(ORIG_LOBS):
    for g_idx, gc in enumerate(year_centers):
        ax.text(gc + offsets[l_idx], -0.005,
                lob["label"],
                ha="right", va="top",
                fontsize=8, color=CPX_DGREY,
                rotation=45,
                transform=ax.get_xaxis_transform())

# Year labels
ax.set_xticks(year_centers)
ax.set_xticklabels(years_avail, fontsize=12, fontweight="bold")
ax.tick_params(axis='x', pad=-450)

ax.set_ylabel("Media Attendance Contribution (M)")
ax.set_title(
    "Annual Media Contribution by Original LOB (Final Model)\n"
    "Grouped by year  |  Media effect only",
    fontsize=12, fontweight="bold", color=CPX_NAVY,
)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.2f}M"))

# Legend — funnel level only
ax.legend(
    handles=[legend_handles[f] for f in ["Awareness", "Consideration", "Conversion"]],
    labels=["Awareness", "Consideration", "Conversion"],
    fontsize=9, ncol=1,
    framealpha=0.9, edgecolor=CPX_LGREY,
    loc="upper left",
    bbox_to_anchor=(0.9, 1),
    borderaxespad=0,
)

ax.set_xlim(year_centers[0] - 0.8, year_centers[-1] + 0.8)
ax.set_ylim(0, ax.get_ylim()[1] * 1.2)
ax.grid(False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()
save_fig(fig, "I2_B9_Annual_OrigLOB_Grouped.png")
plt.show()

In [0]:
# ── CELL: Monthly LOB contribution — merged and original (self-contained) ─────

import matplotlib.ticker as mticker

# ── LOB group mapping ─────────────────────────────────────────────────────────
LOB_GROUP_MAP = {
    "Premium_Brand" : [
        "Premium Experiences - AVX",
        "Premium Experiences - VIP",
        "F&B",
        "EXH - Brand",
    ],
    "Film_Marketing": [
        "Premium Experiences - IMAX",
        "Film Marketing - Inform",
        "Performance - Convert & Upgrade",
    ],
    "Prepaid"       : ["Prepaid - B2B", "Prepaid - B2C"],
    "Cineclub"      : ["Cineclub"],
    "GE"            : ["G&E"],
}

LOB_COLORS = {
    "Awareness__Premium_Brand"    : "#1a3a5c",
    "Consideration__Premium_Brand": "#c0392b",
    "Consideration__GE"           : "#e74c3c",
    "Consideration__Prepaid"      : "#f1948a",
    "Conversion__Film_Marketing"  : "#f5a800",
    "Conversion__Premium_Brand"   : "#c47d00",
    "Conversion__Cineclub"        : "#ffd966",
}

LOB_LABELS = {
    "Awareness__Premium_Brand"    : "Awareness — Premium & Brand",
    "Consideration__Premium_Brand": "Consideration — Premium & Brand",
    "Consideration__GE"           : "Consideration — G&E",
    "Consideration__Prepaid"      : "Consideration — Prepaid",
    "Conversion__Film_Marketing"  : "Conversion — Film Marketing",
    "Conversion__Premium_Brand"   : "Conversion — Premium & Brand",
    "Conversion__Cineclub"        : "Conversion — Cineclub",
}

ORIG_LOB_PALETTE = {
    "Awareness__Premium Experiences - AVX"        : "#0d2137",
    "Awareness__Premium Experiences - VIP"        : "#1a3a5c",
    "Awareness__F&B"                              : "#2e5f8a",
    "Awareness__EXH - Brand"                      : "#4a85b5",
    "Consideration__Premium Experiences - AVX"    : "#7b0e0e",
    "Consideration__Premium Experiences - VIP"    : "#c0392b",
    "Consideration__F&B"                          : "#e74c3c",
    "Consideration__EXH - Brand"                  : "#f1948a",
    "Consideration__G&E"                          : "#922b21",
    "Consideration__Prepaid - B2B"                : "#fadbd8",
    "Consideration__Prepaid - B2C"                : "#f5b7b1",
    "Conversion__Premium Experiences - IMAX"      : "#7d5400",
    "Conversion__Film Marketing - Inform"         : "#c47d00",
    "Conversion__Performance - Convert & Upgrade" : "#f5a800",
    "Conversion__Premium Experiences - AVX"       : "#ffc300",
    "Conversion__Premium Experiences - VIP"       : "#ffd966",
    "Conversion__F&B"                             : "#ffe599",
    "Conversion__EXH - Brand"                     : "#fff2cc",
    "Conversion__Cineclub"                        : "#e6ac00",
}

# ── Step 1: Build original LOB spend columns ──────────────────────────────────
for funnel in ["Awareness", "Consideration", "Conversion"]:
    for lob_group, orig_lobs in LOB_GROUP_MAP.items():
        combo = f"{funnel}__{lob_group}"
        if combo not in ACTIVE_COMBOS:
            continue
        for orig_lob in orig_lobs:
            raw_cols = media_meta.loc[
                (media_meta["funnel"] == funnel) &
                (media_meta["lob"] == orig_lob),
                "col_name"
            ].tolist()
            col_name = f"spend_{funnel}__{orig_lob}"
            df_trainval[col_name] = (
                df.loc[df["week_date"].isin(df_trainval["week_date"]), raw_cols]
                .sum(axis=1).values
                if raw_cols else 0.0
            )

# ── Step 2: Compute spend-share attribution for original LOBs ─────────────────
ORIG_LOB_CONTRIB = {}

for funnel in ["Awareness", "Consideration", "Conversion"]:
    for lob_group, orig_lobs in LOB_GROUP_MAP.items():
        combo = f"{funnel}__{lob_group}"
        if combo not in ACTIVE_COMBOS:
            continue
        combo_contrib = df_trainval[f"media_{combo}"].values
        combo_spend   = df_trainval[combo].values

        for orig_lob in orig_lobs:
            orig_spend  = df_trainval[f"spend_{funnel}__{orig_lob}"].values
            share       = np.where(combo_spend > 0, orig_spend / combo_spend, 0.0)
            contrib_col = f"contrib_{funnel}__{orig_lob}"
            df_trainval[contrib_col] = combo_contrib * share
            ORIG_LOB_CONTRIB[contrib_col] = {
                "funnel"  : funnel,
                "orig_lob": orig_lob,
                "combo"   : combo,
            }

# ── Step 3: Aggregate to month ────────────────────────────────────────────────
df_trainval["month"] = df_trainval["week_date"].dt.to_period("M")

agg_dict = {target_col: "sum"}
for combo in ACTIVE_COMBOS:
    agg_dict[f"media_{combo}"] = "sum"
for col in ORIG_LOB_CONTRIB:
    agg_dict[col] = "sum"

monthly         = df_trainval.groupby("month").agg(agg_dict).reset_index()
monthly["month_dt"] = monthly["month"].dt.to_timestamp()

# Val window helper
val_start  = val_df["week_date"].dt.to_period("M").min().to_timestamp()
val_months = monthly[monthly["month_dt"] >= val_start].index
month_labels = [m.strftime("%b %Y") for m in monthly["month_dt"]]

def add_val_shade(ax):
    if len(val_months) > 0:
        ax.axvspan(val_months[0] - 0.5, val_months[-1] + 0.5,
                   alpha=0.06, color=CPX_RED)
        ax.annotate("← Validation",
                    xy=(val_months[0], ax.get_ylim()[1] * 0.97),
                    fontsize=8, color=CPX_RED, fontstyle="italic")

x = np.arange(len(monthly))
w = 0.65

# ── Chart 1: Merged LOBs ──────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(20, 7))
bottoms = np.zeros(len(monthly))

for combo in ACTIVE_COMBOS:
    vals = monthly[f"media_{combo}"].values / 1e6
    ax.bar(x, vals, width=w, bottom=bottoms,
           color=LOB_COLORS[combo], alpha=0.9,
           label=LOB_LABELS[combo],
           edgecolor="white", linewidth=0.3)
    bottoms += vals

ax.scatter(x, bottoms, color="black", zorder=5,
           s=40, marker="D", label="Total media contribution")
add_val_shade(ax)

ax.set_xticks(x[::3])
ax.set_xticklabels(month_labels[::3], rotation=45, ha="right", fontsize=8)
ax.set_ylabel("Media Attendance Contribution (M)")
ax.set_title(
    "EXH MMM — Monthly Media Contribution by Merged LOB (Final Model)\n"
    "Media effect only — baseline excluded",
    fontsize=12, fontweight="bold", color=CPX_NAVY,
)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.1f}M"))
ax.legend(fontsize=8, ncol=4, loc="upper right",
          framealpha=0.9, edgecolor=CPX_LGREY)
ax.grid(False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
save_fig(fig, "I2_B2_Monthly_MergedLOB_MediaOnly.png")
plt.show()

# ── Chart 2: Original LOBs ────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(22, 8))
bottoms = np.zeros(len(monthly))

for col, meta in ORIG_LOB_CONTRIB.items():
    vals  = monthly[col].values / 1e6
    label = f"{meta['funnel']} — {meta['orig_lob']}"
    color = ORIG_LOB_PALETTE.get(
        f"{meta['funnel']}__{meta['orig_lob']}", "#888888"
    )
    ax.bar(x, vals, width=w, bottom=bottoms,
           color=color, alpha=0.9, label=label,
           edgecolor="white", linewidth=0.3)
    bottoms += vals

ax.scatter(x, bottoms, color="black", zorder=5,
           s=40, marker="D", label="Total media contribution")
add_val_shade(ax)

ax.set_xticks(x[::3])
ax.set_xticklabels(month_labels[::3], rotation=45, ha="right", fontsize=8)
ax.set_ylabel("Media Attendance Contribution (M)")
ax.set_title(
    "EXH MMM — Monthly Media Contribution by Original LOB (Final Model)\n"
    "Media effect only — baseline excluded  |  Within-group attribution proportional to spend share",
    fontsize=12, fontweight="bold", color=CPX_NAVY,
)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.1f}M"))
ax.legend(fontsize=7, ncol=5, loc="upper right",
          framealpha=0.9, edgecolor=CPX_LGREY)
ax.grid(False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
save_fig(fig, "I2_B3_Monthly_OrigLOB_MediaOnly.png")
plt.show()

In [0]:
# ── CELL: Monthly LOB contribution — one bar chart per funnel ─────────────────

FUNNEL_LOB_COLORS = {
    "Awareness": {
        "Awareness__Premium_Brand": "#1a3a5c",
    },
    "Consideration": {
        "Consideration__Premium_Brand": "#c0392b",
        "Consideration__GE"           : "#e74c3c",
        "Consideration__Prepaid"      : "#f1948a",
    },
    "Conversion": {
        "Conversion__Film_Marketing"  : "#f5a800",
        "Conversion__Premium_Brand"   : "#c47d00",
        "Conversion__Cineclub"        : "#ffd966",
    },
}

FUNNEL_LOB_LABELS = {
    "Awareness": {
        "Awareness__Premium_Brand": "Premium & Brand",
    },
    "Consideration": {
        "Consideration__Premium_Brand": "Premium & Brand",
        "Consideration__GE"           : "G&E",
        "Consideration__Prepaid"      : "Prepaid",
    },
    "Conversion": {
        "Conversion__Film_Marketing"  : "Film Marketing",
        "Conversion__Premium_Brand"   : "Premium & Brand",
        "Conversion__Cineclub"        : "Cineclub",
    },
}

x            = np.arange(len(monthly))
w            = 0.65
month_labels = [m.strftime("%b %Y") for m in monthly["month_dt"]]
val_start    = val_df["week_date"].dt.to_period("M").min().to_timestamp()
val_months   = monthly[monthly["month_dt"] >= val_start].index


def add_val_shade(ax):
    if len(val_months) > 0:
        ax.axvspan(val_months[0] - 0.5, val_months[-1] + 0.5,
                   alpha=0.06, color=CPX_RED)
        ax.annotate("← Validation",
                    xy=(val_months[0], ax.get_ylim()[1] * 0.97),
                    fontsize=8, color=CPX_RED, fontstyle="italic")


for funnel in ["Awareness", "Consideration", "Conversion"]:
    combos  = [c for c in ACTIVE_COMBOS if c.startswith(funnel)]
    colors  = FUNNEL_LOB_COLORS[funnel]
    labels  = FUNNEL_LOB_LABELS[funnel]

    fig, ax = plt.subplots(figsize=(20, 7))
    bottoms = np.zeros(len(monthly))

    for combo in combos:
        vals = monthly[f"media_{combo}"].values / 1e6
        ax.bar(x, vals, width=w, bottom=bottoms,
               color=colors[combo], alpha=0.9,
               label=labels[combo],
               edgecolor="white", linewidth=0.3)
        bottoms += vals

    ax.scatter(x, bottoms, color="black", zorder=5,
               s=40, marker="D", label="Total media contribution")

    add_val_shade(ax)

    ax.set_xticks(x[::3])
    ax.set_xticklabels(month_labels[::3], rotation=45, ha="right", fontsize=8)
    ax.set_ylabel("Media Attendance Contribution (M)")
    ax.set_title(
        f"EXH MMM — Monthly Media Contribution: {funnel} Funnel (Final Model)\n"
        "Media effect only — baseline excluded",
        fontsize=12, fontweight="bold", color=CPX_NAVY,
    )
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.2f}M"))
    ax.legend(fontsize=8, ncol=3, loc="upper right",
              framealpha=0.9, edgecolor=CPX_LGREY)
    ax.grid(False)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    plt.tight_layout()
    save_fig(fig, f"I2_B4_{funnel}_LOB_MediaOnly.png")
    plt.show()

In [0]:
# ── CELL: Monthly media contribution by original LOB — one chart per funnel ───

ORIG_LOB_COLORS_BY_FUNNEL = {
    "Awareness": {
        "Premium Experiences - AVX": "#0d2137",
        "Premium Experiences - VIP": "#1a3a5c",
        "F&B"                      : "#2e5f8a",
        "EXH - Brand"              : "#4a85b5",
    },
    "Consideration": {
        "Premium Experiences - AVX": "#7b0e0e",
        "Premium Experiences - VIP": "#c0392b",
        "F&B"                      : "#e74c3c",
        "EXH - Brand"              : "#f1948a",
        "G&E"                      : "#922b21",
        "Prepaid - B2B"            : "#fadbd8",
        "Prepaid - B2C"            : "#f5b7b1",
    },
    "Conversion": {
        "Premium Experiences - IMAX"          : "#7d5400",
        "Film Marketing - Inform"             : "#c47d00",
        "Performance - Convert & Upgrade"     : "#f5a800",
        "Premium Experiences - AVX"           : "#ffc300",
        "Premium Experiences - VIP"           : "#ffd966",
        "F&B"                                 : "#ffe599",
        "EXH - Brand"                         : "#fff2cc",
        "Cineclub"                            : "#e6ac00",
    },
}

x            = np.arange(len(monthly))
w            = 0.65
month_labels = [m.strftime("%b %Y") for m in monthly["month_dt"]]
val_start    = val_df["week_date"].dt.to_period("M").min().to_timestamp()
val_months   = monthly[monthly["month_dt"] >= val_start].index


def add_val_shade(ax):
    if len(val_months) > 0:
        ax.axvspan(val_months[0] - 0.5, val_months[-1] + 0.5,
                   alpha=0.06, color=CPX_RED)
        ax.annotate("← Validation",
                    xy=(val_months[0], ax.get_ylim()[1] * 0.97),
                    fontsize=8, color=CPX_RED, fontstyle="italic")


for funnel in ["Awareness", "Consideration", "Conversion"]:

    # Filter ORIG_LOB_CONTRIB to this funnel only
    funnel_contribs = {
        col: meta for col, meta in ORIG_LOB_CONTRIB.items()
        if meta["funnel"] == funnel
    }

    fig, ax = plt.subplots(figsize=(20, 7))
    bottoms = np.zeros(len(monthly))

    for col, meta in funnel_contribs.items():
        orig_lob = meta["orig_lob"]
        vals     = monthly[col].values / 1e6
        color    = ORIG_LOB_COLORS_BY_FUNNEL[funnel].get(orig_lob, "#888888")

        ax.bar(x, vals, width=w, bottom=bottoms,
               color=color, alpha=0.9,
               label=orig_lob,
               edgecolor="white", linewidth=0.3)
        bottoms += vals

    ax.scatter(x, bottoms, color="black", zorder=5,
               s=40, marker="D", label="Total media contribution")

    add_val_shade(ax)

    ax.set_xticks(x[::3])
    ax.set_xticklabels(month_labels[::3], rotation=45, ha="right", fontsize=8)
    ax.set_ylabel("Media Attendance Contribution (M)")
    ax.set_title(
        f"EXH MMM — Monthly Media Contribution: {funnel} Funnel by LOB (Final Model)\n"
        "Media effect only — baseline excluded",
        fontsize=12, fontweight="bold", color=CPX_NAVY,
    )
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.2f}M"))
    ax.legend(fontsize=8, ncol=4, loc="upper right",
              framealpha=0.9, edgecolor=CPX_LGREY)
    ax.grid(False)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    # Footnote — document approximation
    fig.text(
        0.01, 0.01,
        "* Within-group LOB attribution is proportional to spend share — not independently modelled",
        fontsize=7, color=CPX_DGREY, fontstyle="italic", ha="left"
    )

    plt.tight_layout()
    save_fig(fig, f"I2_B5_{funnel}_OrigLOB_MediaOnly.png")
    plt.show()

In [0]:
# ── CELL: Saturation curves — actual spend vs attendance (per funnel) ─────────

from matplotlib.lines import Line2D

YEAR_LINESTYLES = {
    2023: {"color": "#2ecc71", "linestyle": "-",  "linewidth": 1.2},
    2024: {"color": "#e67e22", "linestyle": "--", "linewidth": 1.2},
    2025: {"color": "#9b59b6", "linestyle": ":",  "linewidth": 1.5},
}

for funnel in ["Awareness", "Consideration", "Conversion"]:

    funnel_orig_lobs = {
        k: v for k, v in active_orig_lobs.items()
        if v["funnel"] == funnel
    }

    n_lobs = len(funnel_orig_lobs)
    ncols  = min(4, n_lobs)
    nrows  = int(np.ceil(n_lobs / ncols))

    fig, axes = plt.subplots(nrows, ncols,
                             figsize=(6 * ncols, 5 * nrows),
                             squeeze=False)
    axes_flat = axes.flatten()
    color     = FUNNEL_COLORS[funnel]

    for idx, (lob_key, lob_info) in enumerate(funnel_orig_lobs.items()):
        ax       = axes_flat[idx]
        orig_lob = lob_info["orig_lob"]
        combo    = lob_info["combo"]
        series   = lob_info["series"]  # normalised spend series

        # ── Get train max spend for this combo ────────────────────────────────
        max_spend = spend_train_max_iter2[combo]

        # ── Saturation params ─────────────────────────────────────────────────
        if funnel == "Consideration":
            k_samp  = post[f"k_{combo}"].values.ravel()[::10]
            as_samp = post[f"alpha_s_{combo}"].values.ravel()[::10]
            sat_samples = np.array([
                hill_sat(x_grid, float(k), float(a))
                for k, a in zip(k_samp, as_samp)
            ])
            k_mean    = float(post[f"k_{combo}"].values.ravel().mean())
            as_mean   = float(post[f"alpha_s_{combo}"].values.ravel().mean())
            sat_pt_norm = saturation_point_hill(k_mean, as_mean)
            param_txt = f"k={k_mean:.3f}, α_s={as_mean:.3f}"
        else:
            lam_samp = post[f"lam_{combo}"].values.ravel()[::10]
            sat_samples = np.array([
                logistic_sat(x_grid, float(l))
                for l in lam_samp
            ])
            lam_mean    = float(post[f"lam_{combo}"].values.ravel().mean())
            sat_pt_norm = saturation_point_logistic(lam_mean)
            param_txt   = f"λ={lam_mean:.3f}"

        # ── Gamma — scale saturation to attendance ────────────────────────────
        gamma_mean = float(post[f"gamma_{combo}"].values.ravel().mean())
        scale      = scaler_y.scale_[0]  # attendance std dev

        # Scale: sat effect → attendance contribution
        sat_mean = sat_samples.mean(axis=0) * gamma_mean * scale
        sat_lo   = np.quantile(sat_samples, 0.025, axis=0) * gamma_mean * scale
        sat_hi   = np.quantile(sat_samples, 0.975, axis=0) * gamma_mean * scale

        # X axis in actual $ spend
        x_spend  = x_grid * max_spend

        # ── Plot curve + CI ───────────────────────────────────────────────────
        ax.plot(x_spend, sat_mean, color=color, linewidth=2.5, zorder=3)
        ax.fill_between(x_spend, sat_lo, sat_hi,
                        color=color, alpha=0.12, zorder=2)

        # ── Train spend range (actual $) ──────────────────────────────────────
        nonzero = series[series > 0] * max_spend
        if len(nonzero) > 0:
            ax.axvspan(nonzero.min(), nonzero.max(),
                       alpha=0.07, color=CPX_DGREY, zorder=1)

        # ── Saturation point ──────────────────────────────────────────────────
        sat_pt_spend = sat_pt_norm * max_spend
        sat_pt_att   = (
            hill_sat(np.array([sat_pt_norm]), k_mean, as_mean)[0]
            if funnel == "Consideration"
            else logistic_sat(np.array([sat_pt_norm]), lam_mean)[0]
        ) * gamma_mean * scale

        if 0 < sat_pt_spend <= x_spend.max():
            ax.axvline(sat_pt_spend, color=color, linewidth=1.0,
                       linestyle=":", alpha=0.7, zorder=4)
            ax.scatter([sat_pt_spend], [sat_pt_att],
                       color=color, zorder=5, s=35, clip_on=False)
            ax.annotate(
                f"  Sat.\n  ${sat_pt_spend:,.0f}",
                xy=(sat_pt_spend, sat_pt_att),
                fontsize=6.5, color=color,
                va="center", ha="left",
            )

        # ── Annual mean spend markers ─────────────────────────────────────────
        for year, style in YEAR_LINESTYLES.items():
            year_mask  = df_trainval["year"] == year
            year_spend = series[year_mask.values]
            if year_spend.sum() == 0:
                continue
            avg_norm    = year_spend.mean()
            avg_spend   = avg_norm * max_spend
            sat_val_att = (
                hill_sat(np.array([avg_norm]), k_mean, as_mean)[0]
                if funnel == "Consideration"
                else logistic_sat(np.array([avg_norm]), lam_mean)[0]
            ) * gamma_mean * scale

            ax.axvline(avg_spend,
                       color=style["color"],
                       linestyle=style["linestyle"],
                       linewidth=style["linewidth"],
                       alpha=0.85, zorder=4)
            ax.scatter([avg_spend], [sat_val_att],
                       color=style["color"], s=20, zorder=5, clip_on=False)
            ax.annotate(
                f"${avg_spend:,.0f}",
                xy=(avg_spend, sat_mean.min() * 0.05),
                fontsize=6, color=style["color"],
                ha="center", va="bottom", rotation=90,
            )

        # ── Formatting ────────────────────────────────────────────────────────
        ax.set_xlabel("Weekly Spend ($)", fontsize=8)
        ax.set_ylabel("Weekly Attendance Contribution", fontsize=8)
        ax.set_title(f"{orig_lob}\n{param_txt}",
                     fontweight="bold", color=CPX_NAVY, fontsize=8)
        ax.xaxis.set_major_formatter(
            plt.FuncFormatter(lambda v, _: f"${v:,.0f}")
        )
        ax.yaxis.set_major_formatter(
            plt.FuncFormatter(lambda v, _: f"{v:,.0f}")
        )
        ax.set_xlim(0, x_spend.max())
        ax.set_ylim(0, sat_hi.max() * 1.1)
        ax.grid(False)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

    # Hide unused axes
    for idx in range(n_lobs, len(axes_flat)):
        axes_flat[idx].set_visible(False)

    # ── Figure-level legend ───────────────────────────────────────────────────
    legend_elements = [
        Line2D([0], [0], color=color, linewidth=2.5,
               label="Response curve (posterior mean)"),
        Line2D([0], [0], color=color, linewidth=6,
               alpha=0.15, label="95% credible interval"),
        Line2D([0], [0], color=CPX_DGREY, linewidth=6,
               alpha=0.15, label="Train spend range"),
        Line2D([0], [0], color=color, linewidth=1,
               linestyle=":", label="95% saturation point"),
    ] + [
        Line2D([0], [0],
               color=style["color"],
               linestyle=style["linestyle"],
               linewidth=style["linewidth"],
               label=f"{year} avg weekly spend")
        for year, style in YEAR_LINESTYLES.items()
    ]

    fig.legend(
        handles=legend_elements,
        fontsize=8, loc="lower center",
        bbox_to_anchor=(0.5, -0.04),
        ncol=4, framealpha=0.9,
        edgecolor=CPX_LGREY,
    )

    sat_type = "Hill" if funnel == "Consideration" else "Logistic"
    fig.suptitle(
        f"Media Response Curves — {funnel} Funnel by Original LOB (Final Model)\n"
        f"X = weekly spend ($)  |  Y = weekly attendance contribution  |  "
        f"Saturation type: {sat_type}",
        fontsize=12, fontweight="bold", color=CPX_NAVY,
    )
    fig.text(
        0.01, 0.01,
        "* Parameters estimated at merged LOB group level — "
        "original LOB curves share shape, spend levels differ by spend-share proportion",
        fontsize=7, color=CPX_DGREY, fontstyle="italic", ha="left",
    )

    plt.tight_layout()
    save_fig(fig, f"I2_D10_ResponseCurve_{funnel}_OrigLOB.png")
    plt.show()

In [0]:
# ── CELL: Total media diminishing returns curve — annual spend markers ─────────

from matplotlib.lines import Line2D
from scipy.optimize import brentq

post       = trace_iter2_mixed_hier.posterior
scale      = scaler_y.scale_[0]
df_trainval["year"] = df_trainval["week_date"].dt.year

FUNNEL_COLORS = {
    "Awareness"    : CPX_NAVY,
    "Consideration": CPX_RED,
    "Conversion"   : CPX_GOLD,
}

YEAR_STYLES = {
    2023: {"color": "#2ecc71", "linestyle": "-",  "linewidth": 1.5, "marker": "o"},
    2024: {"color": "#e67e22", "linestyle": "--", "linewidth": 1.5, "marker": "s"},
    2025: {"color": "#9b59b6", "linestyle": ":",  "linewidth": 1.8, "marker": "D"},
}

# ── Helper functions ──────────────────────────────────────────────────────────
def logistic_sat(x_norm, lam):
    return 1 - np.exp(-lam * x_norm)

def hill_sat(x_norm, k, alpha_s):
    x_safe = np.maximum(x_norm, 1e-6)
    return (x_safe ** alpha_s) / (x_safe ** alpha_s + k ** alpha_s)

def steady_state_adstock(s_norm, alpha):
    return s_norm / (1 - alpha)

def total_media_attendance(weekly_budget_alloc, params):
    """
    weekly_budget_alloc : dict {combo: weekly_spend_$}
    params              : dict of posterior mean parameters
    returns             : total annual media attendance contribution
    """
    total = 0.0
    for combo, weekly_spend in weekly_budget_alloc.items():
        funnel    = combo.split("__")[0]
        max_spend = spend_train_max_iter2[combo]
        alpha     = params[f"alpha_{combo}"]
        gamma     = params[f"gamma_{combo}"]

        s_norm    = weekly_spend / max_spend
        adstock   = steady_state_adstock(s_norm, alpha)

        if funnel == "Consideration":
            k       = params[f"k_{combo}"]
            alpha_s = params[f"alpha_s_{combo}"]
            sat     = hill_sat(adstock, k, alpha_s)
        else:
            lam = params[f"lam_{combo}"]
            sat = logistic_sat(adstock, lam)

        total += gamma * sat * scale * 52  # annualised
    return total

# ── Extract posterior means ───────────────────────────────────────────────────
params_mean = {}
for combo in ACTIVE_COMBOS:
    funnel = combo.split("__")[0]
    params_mean[f"alpha_{combo}"] = float(post[f"alpha_{combo}"].values.ravel().mean())
    params_mean[f"gamma_{combo}"] = float(post[f"gamma_{combo}"].values.ravel().mean())
    if funnel == "Consideration":
        params_mean[f"k_{combo}"]       = float(post[f"k_{combo}"].values.ravel().mean())
        params_mean[f"alpha_s_{combo}"] = float(post[f"alpha_s_{combo}"].values.ravel().mean())
    else:
        params_mean[f"lam_{combo}"] = float(post[f"lam_{combo}"].values.ravel().mean())

# ── 2025 budget shares (proportion of total spend per LOB) ───────────────────
df_2025        = df[df["week_date"].dt.year == 2025]
total_2025_wk  = sum(df_2025[c].mean() for c in ACTIVE_COMBOS)
budget_shares  = {
    c: df_2025[c].mean() / total_2025_wk
    for c in ACTIVE_COMBOS
}

# ── Build total response curve ────────────────────────────────────────────────
# Scale total weekly budget from 0 to 2.5x 2025 weekly
x_budgets  = np.linspace(1, total_2025_wk * 2.5, 300)

# Posterior mean curve
curve_mean = np.array([
    total_media_attendance(
        {c: b * budget_shares[c] for c in ACTIVE_COMBOS},
        params_mean
    )
    for b in x_budgets
])

# 89% CI from 100 posterior samples
n_samples = 100
sample_idx = np.linspace(
    0, post["gamma_Conversion__Film_Marketing"].values.ravel().shape[0] - 1,
    n_samples, dtype=int
)

curve_samples = []
for i in sample_idx:
    p = {}
    for combo in ACTIVE_COMBOS:
        funnel = combo.split("__")[0]
        p[f"alpha_{combo}"] = float(post[f"alpha_{combo}"].values.ravel()[i])
        p[f"gamma_{combo}"] = float(post[f"gamma_{combo}"].values.ravel()[i])
        if funnel == "Consideration":
            p[f"k_{combo}"]       = float(post[f"k_{combo}"].values.ravel()[i])
            p[f"alpha_s_{combo}"] = float(post[f"alpha_s_{combo}"].values.ravel()[i])
        else:
            p[f"lam_{combo}"] = float(post[f"lam_{combo}"].values.ravel()[i])

    curve_samples.append(np.array([
        total_media_attendance(
            {c: b * budget_shares[c] for c in ACTIVE_COMBOS}, p
        )
        for b in x_budgets
    ]))

curve_samples = np.array(curve_samples)
curve_lo      = np.quantile(curve_samples, 0.055, axis=0)
curve_hi      = np.quantile(curve_samples, 0.945, axis=0)

# ── Annual spend and attendance per year ──────────────────────────────────────
year_points = {}
for year in [2023, 2024, 2025]:
    df_yr       = df[df["week_date"].dt.year == year]
    annual_spend = sum(df_yr[c].sum() for c in ACTIVE_COMBOS)
    weekly_avg   = annual_spend / 52
    annual_att   = total_media_attendance(
        {c: df_yr[c].mean() for c in ACTIVE_COMBOS},
        params_mean
    )
    year_points[year] = {
        "annual_spend" : annual_spend,
        "weekly_avg"   : weekly_avg,
        "annual_att"   : annual_att,
    }

# ── Marginal return curve (derivative) ───────────────────────────────────────
dx            = x_budgets[1] - x_budgets[0]
marginal_mean = np.gradient(curve_mean, dx)

# ── Find diminishing returns inflection point (2nd derivative = 0) ────────────
second_deriv  = np.gradient(marginal_mean, dx)
# Find where second derivative crosses zero from negative (inflection)
inflection_idx = None
for i in range(1, len(second_deriv)):
    if second_deriv[i - 1] > 0 and second_deriv[i] <= 0:
        inflection_idx = i
        break

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(16, 12),
                          gridspec_kw={"height_ratios": [2, 1]})

# ── Panel 1: Total response curve ─────────────────────────────────────────────
ax1 = axes[0]

ax1.plot(x_budgets / 1e6, curve_mean / 1e6,
         color=CPX_NAVY, linewidth=2.5,
         label="Total media response (posterior mean)")
ax1.fill_between(x_budgets / 1e6, curve_lo / 1e6, curve_hi / 1e6,
                 color=CPX_NAVY, alpha=0.12, label="89% credible interval")

# Inflection point
if inflection_idx:
    inf_spend = x_budgets[inflection_idx]
    inf_att   = curve_mean[inflection_idx]
    ax1.axvline(inf_spend / 1e6, color=CPX_DGREY,
                linewidth=1.0, linestyle="--", alpha=0.7)
    ax1.scatter([inf_spend / 1e6], [inf_att / 1e6],
                color=CPX_DGREY, s=80, zorder=6, marker="^")
    ax1.annotate(
        f"Inflection\n${inf_spend/1e6:.2f}M/wk",
        xy=(inf_spend / 1e6, inf_att / 1e6),
        xytext=(12, -30), textcoords="offset points",
        fontsize=7.5, color=CPX_DGREY, fontweight="bold",
        arrowprops=dict(arrowstyle="->", color=CPX_DGREY, lw=0.8),
        bbox=dict(boxstyle="round,pad=0.2", fc="white", alpha=0.85, ec="none"),
    )

# Annual spend markers
for year, style in YEAR_STYLES.items():
    if year not in year_points:
        continue
    pts = year_points[year]
    ax1.axvline(pts["weekly_avg"] / 1e6,
                color=style["color"],
                linestyle=style["linestyle"],
                linewidth=style["linewidth"],
                alpha=0.85, zorder=4)
    ax1.scatter([pts["weekly_avg"] / 1e6], [pts["annual_att"] / 1e6],
                color=style["color"], marker=style["marker"],
                s=90, zorder=6)
    ax1.annotate(
        f"{year}\n${pts['annual_spend']/1e6:.1f}M/yr\n"
        f"({pts['annual_att']/1e6:.1f}M att.)",
        xy=(pts["weekly_avg"] / 1e6, pts["annual_att"] / 1e6),
        xytext=(10, 15 - list(YEAR_STYLES.keys()).index(year) * 40),
        textcoords="offset points",
        fontsize=7.5, color=style["color"], fontweight="bold",
        arrowprops=dict(arrowstyle="->", color=style["color"], lw=0.8),
        bbox=dict(boxstyle="round,pad=0.2", fc="white", alpha=0.85, ec="none"),
    )

ax1.set_ylabel("Annual Media Attendance Contribution (M)", fontsize=10)
ax1.set_title(
    "Total Media Response Curve — EXH MMM Final Model\n"
    "Steady-state adstock  |  2025 LOB budget shares held constant  |  "
    "89% credible interval",
    fontsize=12, fontweight="bold", color=CPX_NAVY,
)
ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.1f}M"))
ax1.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"${v:.2f}M"))
ax1.set_xlabel("Weekly Media Budget ($M)", fontsize=10)
ax1.grid(False)
ax1.spines["top"].set_visible(False)
ax1.spines["right"].set_visible(False)

# ── Panel 2: Marginal return curve ────────────────────────────────────────────
ax2 = axes[1]

ax2.plot(x_budgets / 1e6, marginal_mean,
         color=CPX_RED, linewidth=2.0,
         label="Marginal attendance per additional $1/week")
ax2.axhline(0, color="black", linewidth=0.8)

# Inflection on marginal panel
if inflection_idx:
    ax2.axvline(inf_spend / 1e6, color=CPX_DGREY,
                linewidth=1.0, linestyle="--", alpha=0.7,
                label=f"Inflection (${inf_spend/1e6:.2f}M/wk)")

# Year markers on marginal panel
for year, style in YEAR_STYLES.items():
    if year not in year_points:
        continue
    pts       = year_points[year]
    marg_val  = np.interp(pts["weekly_avg"], x_budgets, marginal_mean)
    ax2.axvline(pts["weekly_avg"] / 1e6,
                color=style["color"],
                linestyle=style["linestyle"],
                linewidth=style["linewidth"],
                alpha=0.85)
    ax2.scatter([pts["weekly_avg"] / 1e6], [marg_val],
                color=style["color"], marker=style["marker"], s=70, zorder=5)
    ax2.annotate(
        f"{year}: {marg_val:,.0f} att/$",
        xy=(pts["weekly_avg"] / 1e6, marg_val),
        xytext=(8, 8), textcoords="offset points",
        fontsize=7, color=style["color"], fontweight="bold",
        bbox=dict(boxstyle="round,pad=0.2", fc="white", alpha=0.85, ec="none"),
    )

ax2.set_ylabel("Marginal Attendees per\nAdditional $1/week", fontsize=9)
ax2.set_xlabel("Weekly Media Budget ($M)", fontsize=10)
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:,.0f}"))
ax2.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"${v:.2f}M"))
ax2.grid(False)
ax2.spines["top"].set_visible(False)
ax2.spines["right"].set_visible(False)

# ── Shared legend ─────────────────────────────────────────────────────────────
year_legend = [
    Line2D([0], [0],
           color=style["color"],
           linestyle=style["linestyle"],
           linewidth=style["linewidth"],
           marker=style["marker"],
           markersize=7,
           label=f"{year} (${year_points[year]['annual_spend']/1e6:.1f}M annual)")
    for year, style in YEAR_STYLES.items()
    if year in year_points
]
fig.legend(
    handles=year_legend,
    fontsize=8, loc="lower center",
    bbox_to_anchor=(0.5, -0.02),
    ncol=3, framealpha=0.9, edgecolor=CPX_LGREY,
)

fig.text(
    0.01, 0.01,
    "* Budget distributed across LOBs using 2025 actual spend shares. "
    "Steady-state adstock applied per LOB (1/(1-α)).",
    fontsize=7, color=CPX_DGREY, fontstyle="italic", ha="left",
)

plt.tight_layout()
save_fig(fig, "I2_B10_Total_Media_Response_Curve.png")
plt.show()

# ── Print summary table ───────────────────────────────────────────────────────
print("\nAnnual Spend vs Media Attendance — Year Summary")
print(f"{'Year':<8} {'Annual Spend':>14} {'Weekly Avg':>12} "
      f"{'Media Attend (M)':>18} {'Marginal att/$':>16}")
print("─" * 70)
for year, pts in year_points.items():
    marg = np.interp(pts["weekly_avg"], x_budgets, marginal_mean)
    print(f"{year:<8} ${pts['annual_spend']:>12,.0f} "
          f"${pts['weekly_avg']:>10,.0f} "
          f"{pts['annual_att']/1e6:>17.2f}M "
          f"{marg:>15,.1f}")

if inflection_idx:
    print(f"\nInflection point (max marginal efficiency):")
    print(f"  Weekly budget : ${inf_spend:>10,.0f}")
    print(f"  Annual budget : ${inf_spend*52:>10,.0f}")
    print(f"  Marginal rate : {marginal_mean[inflection_idx]:>10,.1f} att/$")

In [0]:
# ── CELL: Total media diminishing returns curve — annual spend markers ─────────

from matplotlib.lines import Line2D

post       = trace_iter2_mixed_hier.posterior
scale      = scaler_y.scale_[0]
df_trainval["year"] = df_trainval["week_date"].dt.year

FUNNEL_COLORS = {
    "Awareness"    : CPX_NAVY,
    "Consideration": CPX_RED,
    "Conversion"   : CPX_GOLD,
}

YEAR_STYLES = {
    2023: {"color": "#2ecc71", "linestyle": "-",  "linewidth": 1.5, "marker": "o"},
    2024: {"color": "#e67e22", "linestyle": "--", "linewidth": 1.5, "marker": "s"},
    2025: {"color": "#9b59b6", "linestyle": ":",  "linewidth": 1.8, "marker": "D"},
}

# ── Helper functions ──────────────────────────────────────────────────────────
def logistic_sat(x_norm, lam):
    return 1 - np.exp(-lam * x_norm)

def hill_sat(x_norm, k, alpha_s):
    x_safe = np.maximum(x_norm, 1e-6)
    return (x_safe ** alpha_s) / (x_safe ** alpha_s + k ** alpha_s)

def steady_state_adstock(s_norm, alpha):
    return s_norm / (1 - alpha)

def total_media_attendance(weekly_budget_alloc, params):
    total = 0.0
    for combo, weekly_spend in weekly_budget_alloc.items():
        funnel    = combo.split("__")[0]
        max_spend = spend_train_max_iter2[combo]
        alpha     = params[f"alpha_{combo}"]
        gamma     = params[f"gamma_{combo}"]
        s_norm    = weekly_spend / max_spend
        adstock   = steady_state_adstock(s_norm, alpha)
        if funnel == "Consideration":
            k       = params[f"k_{combo}"]
            alpha_s = params[f"alpha_s_{combo}"]
            sat     = hill_sat(adstock, k, alpha_s)
        else:
            lam = params[f"lam_{combo}"]
            sat = logistic_sat(adstock, lam)
        total += gamma * sat * scale * 52
    return total

# ── Extract posterior means ───────────────────────────────────────────────────
params_mean = {}
for combo in ACTIVE_COMBOS:
    funnel = combo.split("__")[0]
    params_mean[f"alpha_{combo}"] = float(post[f"alpha_{combo}"].values.ravel().mean())
    params_mean[f"gamma_{combo}"] = float(post[f"gamma_{combo}"].values.ravel().mean())
    if funnel == "Consideration":
        params_mean[f"k_{combo}"]       = float(post[f"k_{combo}"].values.ravel().mean())
        params_mean[f"alpha_s_{combo}"] = float(post[f"alpha_s_{combo}"].values.ravel().mean())
    else:
        params_mean[f"lam_{combo}"] = float(post[f"lam_{combo}"].values.ravel().mean())

# ── 2025 budget shares ────────────────────────────────────────────────────────
df_2025       = df[df["week_date"].dt.year == 2025]
total_2025_wk = sum(df_2025[c].mean() for c in ACTIVE_COMBOS)
budget_shares = {
    c: df_2025[c].mean() / total_2025_wk
    for c in ACTIVE_COMBOS
}

# ── Budget grid — extended to 10x 2025 weekly to show saturation ──────────────
x_budgets  = np.linspace(1, total_2025_wk * 10, 300)

# ── Posterior mean curve ──────────────────────────────────────────────────────
curve_mean = np.array([
    total_media_attendance(
        {c: b * budget_shares[c] for c in ACTIVE_COMBOS},
        params_mean
    )
    for b in x_budgets
])

# ── 89% CI from 100 posterior samples ────────────────────────────────────────
n_samples  = 100
sample_idx = np.linspace(
    0, post["gamma_Conversion__Film_Marketing"].values.ravel().shape[0] - 1,
    n_samples, dtype=int
)

curve_samples = []
for i in sample_idx:
    p = {}
    for combo in ACTIVE_COMBOS:
        funnel = combo.split("__")[0]
        p[f"alpha_{combo}"] = float(post[f"alpha_{combo}"].values.ravel()[i])
        p[f"gamma_{combo}"] = float(post[f"gamma_{combo}"].values.ravel()[i])
        if funnel == "Consideration":
            p[f"k_{combo}"]       = float(post[f"k_{combo}"].values.ravel()[i])
            p[f"alpha_s_{combo}"] = float(post[f"alpha_s_{combo}"].values.ravel()[i])
        else:
            p[f"lam_{combo}"] = float(post[f"lam_{combo}"].values.ravel()[i])
    curve_samples.append(np.array([
        total_media_attendance(
            {c: b * budget_shares[c] for c in ACTIVE_COMBOS}, p
        )
        for b in x_budgets
    ]))

curve_samples = np.array(curve_samples)
curve_lo      = np.quantile(curve_samples, 0.055, axis=0)
curve_hi      = np.quantile(curve_samples, 0.945, axis=0)

# ── Annual spend and attendance per year ──────────────────────────────────────
year_points = {}
for year in [2023, 2024, 2025]:
    df_yr        = df[df["week_date"].dt.year == year]
    annual_spend = sum(df_yr[c].sum() for c in ACTIVE_COMBOS)
    weekly_avg   = annual_spend / 52
    annual_att   = total_media_attendance(
        {c: df_yr[c].mean() for c in ACTIVE_COMBOS},
        params_mean
    )
    year_points[year] = {
        "annual_spend": annual_spend,
        "weekly_avg"  : weekly_avg,
        "annual_att"  : annual_att,
    }

# ── Marginal return curve ─────────────────────────────────────────────────────
dx            = x_budgets[1] - x_budgets[0]
marginal_mean = np.gradient(curve_mean, dx)

# ── Inflection point (2nd derivative = 0) ────────────────────────────────────
second_deriv   = np.gradient(marginal_mean, dx)
inflection_idx = None
for i in range(1, len(second_deriv)):
    if second_deriv[i - 1] > 0 and second_deriv[i] <= 0:
        inflection_idx = i
        break

# ── 95% saturation point ──────────────────────────────────────────────────────
curve_max    = curve_mean[-1]
sat_95_val   = curve_max * 0.95
sat_95_idx   = np.argmin(np.abs(curve_mean - sat_95_val))
sat_95_spend = x_budgets[sat_95_idx]
sat_95_att   = curve_mean[sat_95_idx]

# ── Plot Panel 1: Total response curve (16:9) ─────────────────────────────────
fig1, ax1 = plt.subplots(figsize=(16, 9))

ax1.plot(x_budgets / 1e6, curve_mean / 1e6,
         color=CPX_NAVY, linewidth=2.5,
         label="Total media response (posterior mean)")
ax1.fill_between(x_budgets / 1e6, curve_lo / 1e6, curve_hi / 1e6,
                 color=CPX_NAVY, alpha=0.12, label="89% credible interval")

# Inflection point
if inflection_idx:
    inf_spend = x_budgets[inflection_idx]
    inf_att   = curve_mean[inflection_idx]
    ax1.axvline(inf_spend / 1e6, color=CPX_DGREY,
                linewidth=1.0, linestyle="--", alpha=0.7)
    ax1.scatter([inf_spend / 1e6], [inf_att / 1e6],
                color=CPX_DGREY, s=80, zorder=6, marker="^")
    ax1.annotate(
        f"Inflection\n${inf_spend/1e6:.2f}M/wk\n(${inf_spend*52/1e6:.1f}M/yr)",
        xy=(inf_spend / 1e6, inf_att / 1e6),
        xytext=(12, -40), textcoords="offset points",
        fontsize=8, color=CPX_DGREY, fontweight="bold",
        arrowprops=dict(arrowstyle="->", color=CPX_DGREY, lw=0.8),
        bbox=dict(boxstyle="round,pad=0.2", fc="white", alpha=0.85, ec="none"),
    )

# 95% saturation point
ax1.axvline(sat_95_spend / 1e6, color=CPX_RED,
            linewidth=1.0, linestyle=":", alpha=0.8)
ax1.scatter([sat_95_spend / 1e6], [sat_95_att / 1e6],
            color=CPX_RED, s=100, zorder=6, marker="*")
ax1.annotate(
    f"95% Saturation\n${sat_95_spend/1e6:.2f}M/wk\n"
    f"(${sat_95_spend*52/1e6:.1f}M/yr)\n"
    f"{sat_95_spend/year_points[2025]['weekly_avg']:.1f}x 2025 spend",
    xy=(sat_95_spend / 1e6, sat_95_att / 1e6),
    xytext=(-80, -50), textcoords="offset points",
    fontsize=8, color=CPX_RED, fontweight="bold",
    arrowprops=dict(arrowstyle="->", color=CPX_RED, lw=0.8),
    bbox=dict(boxstyle="round,pad=0.2", fc="white", alpha=0.85, ec="none"),
)

# Annual spend markers
for year, style in YEAR_STYLES.items():
    if year not in year_points:
        continue
    pts = year_points[year]
    ax1.axvline(pts["weekly_avg"] / 1e6,
                color=style["color"],
                linestyle=style["linestyle"],
                linewidth=style["linewidth"],
                alpha=0.85, zorder=4)
    ax1.scatter([pts["weekly_avg"] / 1e6], [pts["annual_att"] / 1e6],
                color=style["color"], marker=style["marker"],
                s=90, zorder=6)
    ax1.annotate(
        f"{year}\n${pts['annual_spend']/1e6:.1f}M/yr\n"
        f"({pts['annual_att']/1e6:.1f}M att.)",
        xy=(pts["weekly_avg"] / 1e6, pts["annual_att"] / 1e6),
        xytext=(10, 15 - list(YEAR_STYLES.keys()).index(year) * 40),
        textcoords="offset points",
        fontsize=8, color=style["color"], fontweight="bold",
        arrowprops=dict(arrowstyle="->", color=style["color"], lw=0.8),
        bbox=dict(boxstyle="round,pad=0.2", fc="white", alpha=0.85, ec="none"),
    )

ax1.set_ylabel("Annual Media Attendance Contribution (M)", fontsize=11)
ax1.set_xlabel("Weekly Media Budget ($M)", fontsize=11)
ax1.set_title(
    "Total Media Response Curve — EXH MMM Final Model\n"
    "Steady-state adstock  |  2025 LOB budget shares held constant  |  "
    "89% credible interval",
    fontsize=13, fontweight="bold", color=CPX_NAVY,
)
ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.1f}M"))
ax1.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"${v:.2f}M"))

year_legend1 = [
    Line2D([0], [0],
           color=style["color"],
           linestyle=style["linestyle"],
           linewidth=style["linewidth"],
           marker=style["marker"],
           markersize=7,
           label=f"{year} (${year_points[year]['annual_spend']/1e6:.1f}M annual)")
    for year, style in YEAR_STYLES.items()
    if year in year_points
]
ax1.legend(handles=year_legend1, fontsize=9, loc="lower right",
           framealpha=0.9, edgecolor=CPX_LGREY)

ax1.grid(False)
ax1.spines["top"].set_visible(False)
ax1.spines["right"].set_visible(False)

fig1.text(
    0.01, 0.01,
    "* Budget distributed across LOBs using 2025 actual spend shares. "
    "Steady-state adstock applied per LOB (1/(1-α)).",
    fontsize=7, color=CPX_DGREY, fontstyle="italic", ha="left",
)

plt.tight_layout()
save_fig(fig1, "I2_B10a_Total_Media_Response_Curve.png")
plt.show()

# ── Plot Panel 2: Marginal return curve (16:9) ────────────────────────────────
fig2, ax2 = plt.subplots(figsize=(16, 9))

ax2.plot(x_budgets / 1e6, marginal_mean,
         color=CPX_RED, linewidth=2.5,
         label="Marginal attendance per additional $1/week")
ax2.axhline(0, color="black", linewidth=0.8)

if inflection_idx:
    ax2.axvline(inf_spend / 1e6, color=CPX_DGREY,
                linewidth=1.0, linestyle="--", alpha=0.7,
                label=f"Inflection (${inf_spend/1e6:.2f}M/wk)")

ax2.axvline(sat_95_spend / 1e6, color=CPX_RED,
            linewidth=1.0, linestyle=":", alpha=0.8,
            label=f"95% saturation (${sat_95_spend/1e6:.2f}M/wk)")

for year, style in YEAR_STYLES.items():
    if year not in year_points:
        continue
    pts      = year_points[year]
    marg_val = np.interp(pts["weekly_avg"], x_budgets, marginal_mean)
    ax2.axvline(pts["weekly_avg"] / 1e6,
                color=style["color"],
                linestyle=style["linestyle"],
                linewidth=style["linewidth"],
                alpha=0.85)
    ax2.scatter([pts["weekly_avg"] / 1e6], [marg_val],
                color=style["color"], marker=style["marker"], s=80, zorder=5)
    ax2.annotate(
        f"{year}: {marg_val:,.0f} att/$",
        xy=(pts["weekly_avg"] / 1e6, marg_val),
        xytext=(10, 10), textcoords="offset points",
        fontsize=8, color=style["color"], fontweight="bold",
        bbox=dict(boxstyle="round,pad=0.2", fc="white", alpha=0.85, ec="none"),
    )

year_legend2 = [
    Line2D([0], [0],
           color=style["color"],
           linestyle=style["linestyle"],
           linewidth=style["linewidth"],
           marker=style["marker"],
           markersize=7,
           label=f"{year} (${year_points[year]['annual_spend']/1e6:.1f}M annual)")
    for year, style in YEAR_STYLES.items()
    if year in year_points
]

ax2.set_ylabel("Marginal Attendees per Additional $1/week", fontsize=11)
ax2.set_xlabel("Weekly Media Budget ($M)", fontsize=11)
ax2.set_title(
    "Marginal Return Curve — EXH MMM Final Model\n"
    "Incremental attendees generated per additional $1 of weekly media spend",
    fontsize=13, fontweight="bold", color=CPX_NAVY,
)
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:,.0f}"))
ax2.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"${v:.2f}M"))
ax2.legend(handles=year_legend2 + ax2.get_legend_handles_labels()[0][1:],
           fontsize=9, loc="upper right",
           framealpha=0.9, edgecolor=CPX_LGREY)
ax2.grid(False)
ax2.spines["top"].set_visible(False)
ax2.spines["right"].set_visible(False)

fig2.text(
    0.01, 0.01,
    "* Budget distributed across LOBs using 2025 actual spend shares. "
    "Steady-state adstock applied per LOB (1/(1-α)).",
    fontsize=7, color=CPX_DGREY, fontstyle="italic", ha="left",
)

plt.tight_layout()
save_fig(fig2, "I2_B10b_Marginal_Return_Curve.png")
plt.show()

# ── Print summary table ───────────────────────────────────────────────────────
print("\nAnnual Spend vs Media Attendance — Year Summary")
print(f"{'Year':<8} {'Annual Spend':>14} {'Weekly Avg':>12} "
      f"{'Media Attend (M)':>18} {'Marginal att/$':>16}")
print("─" * 70)
for year, pts in year_points.items():
    marg = np.interp(pts["weekly_avg"], x_budgets, marginal_mean)
    print(f"{year:<8} ${pts['annual_spend']:>12,.0f} "
          f"${pts['weekly_avg']:>10,.0f} "
          f"{pts['annual_att']/1e6:>17.2f}M "
          f"{marg:>15,.1f}")

if inflection_idx:
    print(f"\nInflection point:")
    print(f"  Weekly budget  : ${inf_spend:>12,.0f}")
    print(f"  Annual budget  : ${inf_spend*52:>12,.0f}")
    print(f"  Marginal rate  : {marginal_mean[inflection_idx]:>12,.1f} att/$")

print(f"\n95% Saturation point:")
print(f"  Weekly budget  : ${sat_95_spend:>12,.0f}")
print(f"  Annual budget  : ${sat_95_spend*52:>12,.0f}")
print(f"  Annual att.    : {sat_95_att/1e6:>11.2f}M")
print(f"  vs 2025 spend  : {sat_95_spend/year_points[2025]['weekly_avg']:>11.1f}x current weekly")

In [0]:
# ── Run this once in Cell 1 alongside the other pip installs ──────────────────
%pip install h5netcdf

In [0]:
# ── CELL 34: Save model artifacts for Budget Optimiser ───────────────────────
# Run this once after final model is confirmed.
# The optimiser notebook loads everything from ARTIFACT_PATH.

import pickle
import arviz as az
import os

ARTIFACT_PATH = "/Volumes/data_exploration_hub/marketing_hub/model_data/Final_Data/MMM_Artifacts/"
os.makedirs(ARTIFACT_PATH, exist_ok=True)

# ── 1. Save trace ─────────────────────────────────────────────────────────────
trace_iter2_mixed_hier.to_netcdf(
    f"{ARTIFACT_PATH}trace_iter2_mixed_hier.nc"
)
print(f"✓ Trace saved")

# ── 2. Save scalers ───────────────────────────────────────────────────────────
with open(f"{ARTIFACT_PATH}scaler_y.pkl", "wb") as f:
    pickle.dump(scaler_y, f)
with open(f"{ARTIFACT_PATH}scaler_X.pkl", "wb") as f:
    pickle.dump(scaler_X, f)
print(f"✓ Scalers saved")

# ── 3. Save spend metadata ────────────────────────────────────────────────────
with open(f"{ARTIFACT_PATH}spend_train_max_iter2.pkl", "wb") as f:
    pickle.dump(spend_train_max_iter2, f)
print(f"✓ Spend train maxima saved")

# ── 4. Save model structure metadata ─────────────────────────────────────────
model_meta = {
    "ACTIVE_COMBOS"  : ACTIVE_COMBOS,
    "FUNNEL_COMBOS"  : {k: list(v) for k, v in FUNNEL_COMBOS.items()},
    "LOB_GROUP_MAP"  : LOB_GROUP_MAP,
    "ORIG_LOB_CONTRIB": ORIG_LOB_CONTRIB,
    "baseline_features": baseline_features,
    "target_col"     : target_col,
    "n_train"        : n_train,
    "n_val"          : n_val,
    "n_test"         : n_test,
}
with open(f"{ARTIFACT_PATH}model_meta.pkl", "wb") as f:
    pickle.dump(model_meta, f)
print(f"✓ Model metadata saved")

# ── 5. Save 2025 spend baseline ───────────────────────────────────────────────
df_2025 = df[df["week_date"].dt.year == 2025].copy()
baseline_2025 = {}
for combo in ACTIVE_COMBOS:
    baseline_2025[combo] = {
        "annual_spend" : float(df_2025[combo].sum()),
        "weekly_avg"   : float(df_2025[combo].mean()),
        "weeks_active" : int((df_2025[combo] > 0).sum()),
    }
with open(f"{ARTIFACT_PATH}baseline_2025.pkl", "wb") as f:
    pickle.dump(baseline_2025, f)
print(f"✓ 2025 baseline saved")

# ── 6. Save df for backtest (2024 data) ───────────────────────────────────────
df_2024 = df[df["week_date"].dt.year == 2024].copy()
df_2024.to_parquet(f"{ARTIFACT_PATH}df_2024_backtest.parquet", index=False)
df.to_parquet(f"{ARTIFACT_PATH}df_full.parquet", index=False)
print(f"✓ Raw data saved")

# ── Summary ───────────────────────────────────────────────────────────────────
print(f"\n{'='*60}")
print(f"All artifacts saved to: {ARTIFACT_PATH}")
print(f"{'='*60}")
print(f"\nArtifacts:")
for fname in sorted(os.listdir(ARTIFACT_PATH)):
    fsize = os.path.getsize(f"{ARTIFACT_PATH}{fname}") / 1e6
    print(f"  {fname:<45} {fsize:.1f} MB")

print(f"\n2025 Baseline Spend Summary:")
print(f"{'LOB':<35} {'Annual':>12} {'Weekly avg':>12} {'Weeks active':>13}")
print("─" * 74)
total = 0
for combo, vals in baseline_2025.items():
    print(f"{combo:<35} ${vals['annual_spend']:>10,.0f} "
          f"${vals['weekly_avg']:>10,.0f} "
          f"{vals['weeks_active']:>12}")
    total += vals['annual_spend']
print("─" * 74)
print(f"{'TOTAL':<35} ${total:>10,.0f} ${total/52:>10,.0f}")

In [0]:
# ── Do weeks with higher media contribution also have stronger content? ────────

content_cols = [c for c in df_trainval.columns 
                if any(x in c for x in ["cat", "genre", "premium", "tentpole"])]

# Use total_media as proxy for media effectiveness in that week
corr_results = {}
for col in content_cols:
    if df_trainval[col].std() > 0:
        corr = df_trainval["total_media"].corr(df_trainval[col])
        corr_results[col] = corr

corr_df = pd.Series(corr_results).sort_values(ascending=False)
print("Correlation: weekly media contribution vs content variables")
print(corr_df.round(3))

In [0]:
# ── Bin weeks into content tiers and compare media contribution ───────────────

# Use total actual attendance as a proxy for content strength
# (baseline-driven weeks = strong content, residual weeks = weak content)
df_trainval["content_tier"] = pd.qcut(
    df_trainval["controls_pred"],
    q=3,
    labels=["Low content", "Medium content", "High content"]
)

tier_summary = df_trainval.groupby("content_tier").agg(
    avg_actual        = (target_col,     "mean"),
    avg_media_contrib = ("total_media",   "mean"),
    avg_conversion    = ("media_Conversion", "mean"),
    avg_awareness     = ("media_Awareness",  "mean"),
    avg_consideration = ("media_Consideration", "mean"),
    n_weeks           = (target_col,     "count"),
).round(0)

tier_summary["media_pct"] = (
    tier_summary["avg_media_contrib"] / tier_summary["avg_actual"] * 100
).round(1)

print("\nMedia contribution by content tier:")
print(tier_summary[["avg_actual", "avg_media_contrib", "media_pct",
                     "avg_conversion", "avg_awareness", "n_weeks"]])

In [0]:
# ── Does the model under-predict more during high-content weeks? ───────────────
# Under-prediction in high content weeks suggests media effect is amplified
# by content that the model doesn't capture through the interaction

resid_by_tier = df_trainval.groupby("content_tier").agg(
    mean_resid   = ("resid", "mean"),
    std_resid    = ("resid", "std"),
    median_resid = ("resid", lambda x: np.median(x)),
).round(0)

print("\nResiduals by content tier:")
print(resid_by_tier)
print("\nPositive mean residual = model under-predicts = actual > predicted")
print("If high content tier has positive residuals, media effect is amplified by content")

In [0]:
# ── CELL: Full evidence — media contribution vs content ───────────────────────

import warnings
warnings.filterwarnings("ignore")

# ── Content signal columns ────────────────────────────────────────────────────
genre_cols = [
    "action_intensity", "adaptation_intensity", "adventure_intensity",
    "animated_intensity", "biopic_intensity", "comedy_intensity",
    "crime_intensity", "documentary_intensity", "drama_intensity",
    "family_intensity", "fantasy_intensity", "foreign_intensity",
    "horror_intensity", "musical_intensity", "mystery_intensity",
    "period_intensity", "political_intensity", "romance_intensity",
    "romanticcomedy_intensity", "scifi_intensity", "sequel_intensity",
    "suspense_intensity", "thriller_intensity"
]

cat_cols      = ["cat1_share", "cat2_share", "cat3_share", "cat4_share", "cat5_share"]
calendar_cols = ["is_long_weekend", "stat_holiday_in_week",
                 "season_summer", "season_fall", "season_spring", "season_transition"]
volume_cols   = ["active_films"]

all_content_cols = genre_cols + cat_cols + calendar_cols + volume_cols

# ── Evidence 1: Correlation between media contribution and content signals ─────
print("=" * 70)
print("EVIDENCE 1 — Correlation: weekly media contribution vs content signals")
print("=" * 70)

corr_results = {}
for col in all_content_cols:
    if col in df_trainval.columns and df_trainval[col].std() > 0:
        corr_results[col] = df_trainval["total_media"].corr(df_trainval[col])

corr_df = pd.Series(corr_results).sort_values(ascending=False)

print("\nTop 10 positive correlations (content drives more media contribution):")
print(corr_df.head(10).round(3).to_string())
print("\nTop 10 negative correlations (content associated with less media contribution):")
print(corr_df.tail(10).round(3).to_string())

# ── Evidence 2: Media contribution by content tier ────────────────────────────
print("\n" + "=" * 70)
print("EVIDENCE 2 — Media contribution by content tier (controls_pred terciles)")
print("=" * 70)

df_trainval["content_tier_v2"] = pd.qcut(
    df_trainval["controls_pred"],
    q=3,
    labels=["Low content", "Medium content", "High content"],
)

tier_summary = df_trainval.groupby("content_tier_v2", observed=True).agg(
    avg_actual           = (target_col,              "mean"),
    avg_baseline         = ("controls_pred",          "mean"),
    avg_media_total      = ("total_media",            "mean"),
    avg_conversion       = ("media_Conversion",       "mean"),
    avg_awareness        = ("media_Awareness",        "mean"),
    avg_consideration    = ("media_Consideration",    "mean"),
    n_weeks              = (target_col,               "count"),
).round(0)

tier_summary["media_pct"]      = (tier_summary["avg_media_total"]   / tier_summary["avg_actual"] * 100).round(1)
tier_summary["conversion_pct"] = (tier_summary["avg_conversion"]    / tier_summary["avg_actual"] * 100).round(1)
tier_summary["awareness_pct"]  = (tier_summary["avg_awareness"]     / tier_summary["avg_actual"] * 100).round(1)

print(f"\n{'Tier':<18} {'Actual':>10} {'Baseline':>10} {'Media':>10} "
      f"{'Media%':>8} {'Conv%':>8} {'Aware%':>8} {'Weeks':>7}")
print("─" * 75)
for tier, row in tier_summary.iterrows():
    print(f"{str(tier):<18} {row['avg_actual']:>10,.0f} {row['avg_baseline']:>10,.0f} "
          f"{row['avg_media_total']:>10,.0f} {row['media_pct']:>7.1f}% "
          f"{row['conversion_pct']:>7.1f}% {row['awareness_pct']:>7.1f}% "
          f"{row['n_weeks']:>7.0f}")

# ── Evidence 3: Media contribution by active films count ──────────────────────
print("\n" + "=" * 70)
print("EVIDENCE 3 — Media contribution by number of active films")
print("=" * 70)

df_trainval["film_bucket"] = pd.cut(
    df_trainval["active_films"],
    bins=[-1, 5, 10, 15, 100],
    labels=["1-5 films", "6-10 films", "11-15 films", "16+ films"],
)

film_summary = df_trainval.groupby("film_bucket", observed=True).agg(
    avg_actual      = (target_col,   "mean"),
    avg_media       = ("total_media", "mean"),
    avg_conversion  = ("media_Conversion", "mean"),
    n_weeks         = (target_col,   "count"),
).round(0)

film_summary["media_pct"] = (film_summary["avg_media"] / film_summary["avg_actual"] * 100).round(1)

print(f"\n{'Film bucket':<14} {'Actual':>10} {'Media':>10} {'Media%':>8} "
      f"{'Conversion':>12} {'Weeks':>7}")
print("─" * 60)
for bucket, row in film_summary.iterrows():
    print(f"{str(bucket):<14} {row['avg_actual']:>10,.0f} {row['avg_media']:>10,.0f} "
          f"{row['media_pct']:>7.1f}% {row['avg_conversion']:>12,.0f} "
          f"{row['n_weeks']:>7.0f}")

# ── Evidence 4: Residuals by content tier ────────────────────────────────────
print("\n" + "=" * 70)
print("EVIDENCE 4 — Residuals by content tier")
print("(Positive = model under-predicts = media effect amplified by content)")
print("=" * 70)

resid_summary = df_trainval.groupby("content_tier_v2", observed=True).agg(
    mean_resid   = ("resid", "mean"),
    std_resid    = ("resid", "std"),
    median_resid = ("resid", lambda x: np.median(x)),
    pct_positive = ("resid", lambda x: (x > 0).mean() * 100),
).round(0)

print(f"\n{'Tier':<18} {'Mean resid':>12} {'Std resid':>12} "
      f"{'Median resid':>14} {'% positive':>11}")
print("─" * 70)
for tier, row in resid_summary.iterrows():
    print(f"{str(tier):<18} {row['mean_resid']:>12,.0f} {row['std_resid']:>12,.0f} "
          f"{row['median_resid']:>14,.0f} {row['pct_positive']:>10.1f}%")

# ── Evidence 5: Media contribution by season ─────────────────────────────────
print("\n" + "=" * 70)
print("EVIDENCE 5 — Media contribution by season")
print("=" * 70)

season_map = {
    "season_summer"     : "Summer",
    "season_fall"       : "Fall",
    "season_spring"     : "Spring",
    "season_transition" : "Transition",
}

print(f"\n{'Season':<14} {'Avg actual':>12} {'Avg media':>12} "
      f"{'Media%':>8} {'Avg conversion':>16} {'Weeks':>7}")
print("─" * 72)
for col, label in season_map.items():
    sub = df_trainval[df_trainval[col] == 1]
    if len(sub) == 0:
        continue
    avg_actual = sub[target_col].mean()
    avg_media  = sub["total_media"].mean()
    avg_conv   = sub["media_Conversion"].mean()
    n          = len(sub)
    print(f"{label:<14} {avg_actual:>12,.0f} {avg_media:>12,.0f} "
          f"{avg_media/avg_actual*100:>7.1f}% {avg_conv:>16,.0f} {n:>7}")

# Non-season weeks
other = df_trainval[
    (df_trainval["season_summer"] == 0) &
    (df_trainval["season_fall"] == 0) &
    (df_trainval["season_spring"] == 0) &
    (df_trainval["season_transition"] == 0)
]
if len(other) > 0:
    print(f"{'Other/Winter':<14} {other[target_col].mean():>12,.0f} "
          f"{other['total_media'].mean():>12,.0f} "
          f"{other['total_media'].mean()/other[target_col].mean()*100:>7.1f}% "
          f"{other['media_Conversion'].mean():>16,.0f} {len(other):>7}")

# ── Evidence 6: Top genre correlations visualised ─────────────────────────────
print("\n" + "=" * 70)
print("EVIDENCE 6 — Top genre correlations with each funnel")
print("=" * 70)

for funnel_col, label in [
    ("media_Conversion",    "Conversion"),
    ("media_Awareness",     "Awareness"),
    ("media_Consideration", "Consideration"),
]:
    corrs = {
        col: df_trainval[funnel_col].corr(df_trainval[col])
        for col in genre_cols + cat_cols
        if col in df_trainval.columns and df_trainval[col].std() > 0
    }
    top = pd.Series(corrs).sort_values(ascending=False)
    print(f"\n{label} — top 5 positive / top 5 negative:")
    print(f"  Positive: {dict(top.head(5).round(3))}")
    print(f"  Negative: {dict(top.tail(5).round(3))}")

In [0]:
# ── CELL: Film slate strength (cat tiers) vs media contribution ───────────────

# ── Step 1: Understand what cat1-5 represent in the data ─────────────────────
print("=" * 70)
print("CAT TIER DISTRIBUTION — Descriptive Stats")
print("=" * 70)
print(f"\n{'Column':<12} {'Mean':>8} {'Std':>8} {'Min':>8} {'Max':>8} "
      f"{'% zero weeks':>14}")
print("─" * 55)
for col in cat_cols:
    s = df_trainval[col]
    pct_zero = (s == 0).mean() * 100
    print(f"{col:<12} {s.mean():>8.3f} {s.std():>8.3f} "
          f"{s.min():>8.3f} {s.max():>8.3f} {pct_zero:>13.1f}%")

# ── Step 2: Correlations between cat tiers and attendance / media ─────────────
print("\n" + "=" * 70)
print("CAT TIER CORRELATIONS")
print("=" * 70)
print(f"\n{'Column':<12} {'vs Actual':>12} {'vs Total media':>15} "
      f"{'vs Conversion':>15} {'vs Awareness':>13} {'vs Consideration':>17}")
print("─" * 75)
for col in cat_cols:
    c_actual  = df_trainval[col].corr(df_trainval[target_col])
    c_media   = df_trainval[col].corr(df_trainval["total_media"])
    c_conv    = df_trainval[col].corr(df_trainval["media_Conversion"])
    c_aware   = df_trainval[col].corr(df_trainval["media_Awareness"])
    c_consid  = df_trainval[col].corr(df_trainval["media_Consideration"])
    print(f"{col:<12} {c_actual:>12.3f} {c_media:>15.3f} "
          f"{c_conv:>15.3f} {c_aware:>13.3f} {c_consid:>17.3f}")

# ── Step 3: Define slate strength score and bin into tiers ───────────────────
# Cat1 = tentpole/highest tier, Cat5 = lowest tier
# Slate strength = weighted share: cat1 × 5 + cat2 × 4 + cat3 × 3 + cat4 × 2 + cat5 × 1
# Higher score = stronger slate
df_trainval["slate_strength"] = (
    df_trainval["cat1_share"] * 5 +
    df_trainval["cat2_share"] * 4 +
    df_trainval["cat3_share"] * 3 +
    df_trainval["cat4_share"] * 2 +
    df_trainval["cat5_share"] * 1
)

print("\n" + "=" * 70)
print("SLATE STRENGTH SCORE — Distribution")
print("(cat1×5 + cat2×4 + cat3×3 + cat4×2 + cat5×1)")
print("=" * 70)
print(df_trainval["slate_strength"].describe().round(3))

# ── Step 4: Media contribution by slate strength tier ────────────────────────
df_trainval["slate_tier"] = pd.qcut(
    df_trainval["slate_strength"],
    q=4,
    labels=["Weak slate", "Below avg slate", "Above avg slate", "Strong slate"]
)

slate_summary = df_trainval.groupby("slate_tier", observed=True).agg(
    avg_actual        = (target_col,           "mean"),
    avg_media         = ("total_media",         "mean"),
    avg_conversion    = ("media_Conversion",    "mean"),
    avg_awareness     = ("media_Awareness",     "mean"),
    avg_consideration = ("media_Consideration", "mean"),
    avg_cat1          = ("cat1_share",          "mean"),
    avg_cat2          = ("cat2_share",          "mean"),
    n_weeks           = (target_col,            "count"),
).round(3)

slate_summary["media_pct"]      = (slate_summary["avg_media"]      / slate_summary["avg_actual"] * 100).round(1)
slate_summary["conversion_pct"] = (slate_summary["avg_conversion"] / slate_summary["avg_actual"] * 100).round(1)

print("\n" + "=" * 70)
print("MEDIA CONTRIBUTION BY SLATE STRENGTH")
print("=" * 70)
print(f"\n{'Slate tier':<20} {'Actual':>10} {'Media':>10} {'Media%':>8} "
      f"{'Conv%':>8} {'Cat1 share':>12} {'Cat2 share':>12} {'Weeks':>7}")
print("─" * 90)
for tier, row in slate_summary.iterrows():
    print(f"{str(tier):<20} {row['avg_actual']:>10,.0f} {row['avg_media']:>10,.0f} "
          f"{row['media_pct']:>7.1f}% {row['conversion_pct']:>7.1f}% "
          f"{row['avg_cat1']:>12.3f} {row['avg_cat2']:>12.3f} {row['n_weeks']:>7.0f}")

# ── Step 5: Cat1 share specifically — tentpole weeks ─────────────────────────
print("\n" + "=" * 70)
print("TENTPOLE WEEKS (cat1_share > 0) vs NON-TENTPOLE WEEKS")
print("=" * 70)

tentpole    = df_trainval[df_trainval["cat1_share"] > 0]
non_tent    = df_trainval[df_trainval["cat1_share"] == 0]

for label, sub in [("Tentpole weeks", tentpole), ("Non-tentpole weeks", non_tent)]:
    if len(sub) == 0:
        continue
    avg_actual = sub[target_col].mean()
    avg_media  = sub["total_media"].mean()
    avg_conv   = sub["media_Conversion"].mean()
    avg_aware  = sub["media_Awareness"].mean()
    print(f"\n{label} (n={len(sub)}):")
    print(f"  Avg actual attendance : {avg_actual:>12,.0f}")
    print(f"  Avg media contribution: {avg_media:>12,.0f}  ({avg_media/avg_actual*100:.1f}%)")
    print(f"  Avg conversion contrib: {avg_conv:>12,.0f}  ({avg_conv/avg_actual*100:.1f}%)")
    print(f"  Avg awareness contrib : {avg_aware:>12,.0f}  ({avg_aware/avg_actual*100:.1f}%)")

# ── Step 6: Weekly scatter — slate strength vs media contribution ─────────────
print("\n" + "=" * 70)
print("SLATE STRENGTH vs MEDIA CONTRIBUTION — Correlation Summary")
print("=" * 70)
for col, label in [
    ("total_media",         "Total media"),
    ("media_Conversion",    "Conversion"),
    ("media_Awareness",     "Awareness"),
    ("media_Consideration", "Consideration"),
]:
    r = df_trainval["slate_strength"].corr(df_trainval[col])
    print(f"  Slate strength vs {label:<20}: {r:>8.3f}")

In [0]:
# ── Annual media contribution breakdown by year ───────────────────────────────

print("=" * 75)
print("MEDIA CONTRIBUTION BY YEAR — Annual Summary")
print("=" * 75)
print(f"\n{'Year':<6} {'Actual':>12} {'Total media':>13} {'Media%':>8} "
      f"{'Tentpole wks':>14} {'Non-tent wks':>14}")
print("─" * 75)

for year in [2023, 2024, 2025]:
    yr          = df_trainval[df_trainval["year"] == year]
    tent_yr     = yr[yr["cat1_share"] > 0]
    non_tent_yr = yr[yr["cat1_share"] == 0]

    actual      = yr[target_col].sum()
    media       = yr["total_media"].sum()
    tent_media  = tent_yr["total_media"].sum()
    non_media   = non_tent_yr["total_media"].sum()

    print(f"{year:<6} {actual:>12,.0f} {media:>13,.0f} "
          f"{media/actual*100:>7.1f}% "
          f"{len(tent_yr):>8}wks {tent_media/media*100:>5.1f}%  "
          f"{len(non_tent_yr):>8}wks {non_media/media*100:>5.1f}%")

print("─" * 75)
total_actual = df_trainval[target_col].sum()
total_media  = df_trainval["total_media"].sum()
print(f"{'Total':<6} {total_actual:>12,.0f} {total_media:>13,.0f} "
      f"{total_media/total_actual*100:>7.1f}%")

In [0]:
# ── df_full rebuild — scale spend inline from df ─────────────────────────────

df_full = pd.concat([train_df, val_df, test_df]).copy().reset_index(drop=True)
df_full["segment"] = (
    ["train"] * n_train +
    ["val"]   * n_val   +
    ["test"]  * n_test
)
df_full["year"] = df_full["week_date"].dt.year

X_full           = np.vstack([X_train, X_val, X_test])
mu_controls_full = intercept_mean + X_full @ beta_mean
mu_media_full    = np.zeros(n_train + n_val + n_test)
media_contrib_full = {}

for funnel, combos in FUNNEL_COMBOS.items():
    for combo in combos:
        alpha_mean = float(post[f"alpha_{combo}"].values.ravel().mean())
        gamma_mean = float(post[f"gamma_{combo}"].values.ravel().mean())

        # ── Scale full 165-week spend inline ──────────────────────────────────
        max_spend  = spend_train_max_iter2[combo]
        x_spend    = df_full[combo].values.astype(np.float64) / max_spend

        adstocked    = np.zeros(len(x_spend))
        adstocked[0] = x_spend[0]
        for t in range(1, len(x_spend)):
            adstocked[t] = x_spend[t] + alpha_mean * adstocked[t - 1]

        if funnel == "Consideration":
            k_mean   = float(post[f"k_{combo}"].values.ravel().mean())
            as_mean  = float(post[f"alpha_s_{combo}"].values.ravel().mean())
            x_safe   = np.maximum(adstocked, 1e-6)
            sat      = (x_safe ** as_mean) / (x_safe ** as_mean + k_mean ** as_mean)
        else:
            lam_mean = float(post[f"lam_{combo}"].values.ravel().mean())
            sat      = 1 - np.exp(-lam_mean * adstocked)

        contrib                    = gamma_mean * sat * scaler_y.scale_[0]
        mu_media_full             += gamma_mean * sat
        media_contrib_full[combo]  = contrib

df_full["pred"]          = scaler_y.inverse_transform(
                               (mu_controls_full + mu_media_full).reshape(-1, 1)).ravel()
df_full["controls_pred"] = scaler_y.inverse_transform(
                               mu_controls_full.reshape(-1, 1)).ravel()
df_full["total_media"]   = sum(media_contrib_full[c] for c in ACTIVE_COMBOS)

for combo in ACTIVE_COMBOS:
    df_full[f"media_{combo}"] = media_contrib_full[combo]

for funnel, combos in FUNNEL_COMBOS.items():
    df_full[f"media_{funnel}"] = sum(df_full[f"media_{combo}"] for combo in combos)

print(f"✓ df_full built — {len(df_full)} rows  "
      f"{df_full['week_date'].min().date()} to {df_full['week_date'].max().date()}")
print(f"  Train: {n_train}  Val: {n_val}  Test: {n_test}")

# ── Year summary ──────────────────────────────────────────────────────────────
print("\n" + "=" * 75)
print("MEDIA CONTRIBUTION BY YEAR — Full Year (all 165 weeks)")
print("=" * 75)
print(f"\n{'Year':<6} {'Actual':>12} {'Total media':>13} {'Media%':>8} "
      f"{'Tentpole wks':>14} {'Non-tent wks':>14}")
print("─" * 75)

for year in [2023, 2024, 2025]:
    yr          = df_full[df_full["year"] == year]
    tent_yr     = yr[yr["cat1_share"] > 0]
    non_tent_yr = yr[yr["cat1_share"] == 0]
    actual      = yr[target_col].sum()
    media       = yr["total_media"].sum()
    tent_media  = tent_yr["total_media"].sum()
    non_media   = non_tent_yr["total_media"].sum()

    print(f"{year:<6} {actual:>12,.0f} {media:>13,.0f} "
          f"{media/actual*100:>7.1f}% "
          f"{len(tent_yr):>8}wks {tent_media/media*100:>5.1f}%  "
          f"{len(non_tent_yr):>8}wks {non_media/media*100:>5.1f}%")

print("─" * 75)
total_actual = df_full[target_col].sum()
total_media  = df_full["total_media"].sum()
print(f"{'Total':<6} {total_actual:>12,.0f} {total_media:>13,.0f} "
      f"{total_media/total_actual*100:>7.1f}%")

In [0]:
# ── Build X_test from test_df ─────────────────────────────────────────────────
X_test = scaler_X.transform(test_df[baseline_features].values)
print(f"✓ X_test built — shape {X_test.shape}")